# Data cleaning: UniMelb servers and UFCG workloads

This notebook cleans two raw datasets and turns them into the inputs of the
thermal-aware workload scheduling model:

- **UniMelb**, telemetry from physical servers, supplies the server parameters. It comes
  as two datasets from the same cloud, one per server model: Dell PowerEdge **R840** and
  Dell PowerEdge **C6320**.
- **UFCG**, an OpenStack virtual-machine trace, supplies the workloads.

Each section says what a cleaning step does and why it is needed. Each code cell prints
how many rows, servers or VMs the step removed, and the same counts are saved to a
cleaning log, so every figure can be traced back to a step.

The notebook is self-contained: it runs on its own, locally or on Google Colab. All
settings are in the first code cell, and the only other things it needs are the raw
data files listed below.

## Before running

**1. Install the requirements.** Python 3.11 or later with `duckdb`, `numpy` and Jupyter
is enough:

```
pip install duckdb numpy jupyter
```

**2. Put the raw data files in `data/raw/`**, with exactly these names:

```
<project folder>/
├── notebooks/
│   └── data_prep.ipynb      this notebook
└── data/
    └── raw/
        ├── UniMelb.csv
        ├── UniMelb_C6320.csv
        └── UFCG/
            ├── servers_usage.csv
            ├── servers_specs.csv
            └── flavors.csv
```

| File | Size | Columns that are read |
|---|---|---|
| `data/raw/UniMelb.csv` | 613 MB | `Series`, `Time`, `CPU_Load`, `Power`, `Inlet_Temperature`, `CPU1_Temperature`, `CPU2_Temperature`, `Fan_speed1`, `Ram`, `CPU_cores` |
| `data/raw/UniMelb_C6320.csv` | 184 MB | `Series`, `Time`, `CPU_Load`, `Power`, `CPU1_Temperature`, `CPU2_Temperature`, `Ram`, `CPU_cores` |
| `data/raw/UFCG/servers_usage.csv` | 1.94 GB | `timestamp`, `server_id`, `vcpu_usage`, `host_id` |
| `data/raw/UFCG/servers_specs.csv` | 1.50 GB | `server_id`, `flavor_id` |
| `data/raw/UFCG/flavors.csv` | 1.5 kB | `flavor_id`, `flavor_name`, `vcpu`, `ram` |

`UniMelb.csv` is dataset 1 (Dell R840 servers) and `UniMelb_C6320.csv` is dataset 2 (Dell
C6320 servers) of Zenodo record 10.5281/zenodo.10069402 (Ilager et al., UCC 2023), published
there as `dataset1_combined.csv` and `dataset2_combined.csv`.

The five files come to about 4.2 GB. The other files of the UFCG release (another 4.7 GB)
are not needed, so leave them out, especially on Google Drive. The raw files are only
read, never changed. If a file's size differs from the one above, the notebook says so: it
is a different copy of the data, and section 4 will show whether the scenarios differ.

On **Google Colab**, put a folder on Google Drive (by default `My Drive/thermasched/`)
with `data/raw/` inside, laid out as above, and set `COLAB_DRIVE_FOLDER` in the first
code cell if it is named differently. The notebook mounts Drive (Colab asks for
permission) and writes `data/clean/` and `scenarios/` into that folder, ready to be solved. DuckDB is installed if it is missing.

**3. Run every cell from top to bottom.** All paths are relative to the project folder,
which the first code cell finds on its own when Jupyter is started in it or in
`notebooks/` (on Colab, the Drive folder). If the data is somewhere else, set
`ROOT` in the first code cell to the folder that contains `data/raw/`. The notebook stops
with a message naming any raw file it cannot find. A full run takes about a minute on a
laptop. DuckDB reads the large files from disk and is capped at 9 GB of memory; lower
`memory_limit` in `connect()` on a smaller machine.

## What it produces

| Part | Reads | Writes |
|---|---|---|
| 1. Servers | `data/raw/UniMelb.csv`, `UniMelb_C6320.csv` | `data/clean/servers.csv`, `cleaning_log_servers.csv` |
| 2. Workloads | `data/raw/UFCG/servers_usage.csv`, `servers_specs.csv`, `flavors.csv` | `data/clean/workloads.csv`, `workload_usage.parquet`, `cleaning_log_workloads.csv` |
| 3. Model inputs | the two cleaned tables | `scenarios/<name>/`: the 6-server fleet on a light, normal and heavy day, `small` and `small_running` |
| 4. Checks | all outputs | nothing; every assertion must pass, and each scenario is compared with the reference |

Both output folders are created by the notebook. Equation numbers such as "Eq. (17)"
refer to the model formulation in the thesis.

**Run note.** The outputs saved in this copy come from two runs. Parts 1 and 2 and the
file list in part 5 were run with the settings shown, which describe each fleet as a table
of server model, frequency, zone and cold plate (the `fleet` lists). The construction
rules in part 3 still expect the previous settings (`freq_mhz` and `freq_split`), so parts
3 and 4 were not run again: their outputs are those of the previous release, built from
the earlier server table, and running part 3 with these settings stops with an error until
the construction rules are updated for the `fleet` lists.

## Settings

Every threshold used below is a named value in this cell, with the reason for it beside
it. To test a different choice, change the value and run the notebook again.

In [1]:
import sys
from pathlib import Path

# Project folder: the one that contains data/raw/. Jupyter usually starts in notebooks/,
# so the parent is used then. On Colab it is a folder on Google Drive, under "My Drive",
# and Drive is mounted here. Set ROOT by hand if the data is somewhere else.
COLAB_DRIVE_FOLDER = "thermasched"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

cfg = {
    "paths": {
        "unimelb": "data/raw/UniMelb.csv",              # dataset 1: Dell R840 servers
        "unimelb_c6320": "data/raw/UniMelb_C6320.csv",  # dataset 2: Dell C6320 servers
        "ufcg": "data/raw/UFCG",
        "clean": "data/clean",          # created by the notebook
        "scenarios": "scenarios",       # created by the notebook
    },
    # The Dell R840 servers (dataset 1). The unit of `Ram`, the idle threshold and the gates
    # below apply to the C6320 servers too.
    "servers": {
        "model": "R840",
        # UniMelb reports 64 cores until this instant and 128 afterwards (hyper-threads),
        # with a different load scale and idle power. Only the 64-core period is used.
        "regime_end": "2019-03-14 11:40:00",
        "cores": 64,
        # `Ram` is in GB: each Dell R840 host of this dataset has 64 GB of memory (Ilager et
        # al., UCC 2023; dataset D1 of Zenodo record 10.5281/zenodo.10069402), and its
        # largest `Ram_Used` is 64.42. So M_s = 64.488 x 1024 = 66,036 MB. The C6320 hosts
        # report 193.33 (M_s = 197,972 MB), in the same unit.
        "ram_unit_gb": 1.0,
        # CPU_Load is on a 0-100 scale, so the 0.01 threshold of Eq. (17) is 1.0 here.
        "idle_cpu_load": 1.0,
        # Inlet 4.0 degC with fan 280 rpm is a failed-sensor reading.
        "inlet_sentinel": 4.0,
        "fan_sentinel": 280.0,
        # Gates for a server to have a usable power curve, Eq. (17)-(20).
        "min_completeness": 0.90,
        "min_peak_load": 30.0,
        "min_power_load_corr": 0.80,
        "min_idle_rows": 30,
    },
    # The Dell C6320 servers (dataset 2, 1 Feb - 20 Jun 2019). It has no inlet temperature and
    # no fan speeds; the inlet is taken from the R840 room (section 1.6).
    "servers_c6320": {
        "model": "C6320",
        # `CPU_cores` counts the cloud's vCPU slots: 80 in the first hosts' first days, 160
        # from early February, and 320 on some hosts from 15 March 2019, each on its own date.
        # Records above 160 are dropped, as the R840's 128-core period is. About half of the
        # records up to 15 March have no count at all; they are kept.
        "max_slots": 160,
        # Hardware threads: the count before the cloud doubled its vCPU slots (80 -> 160),
        # matching the R840's 64. A decision, since the dataset names no processor.
        "cores": 80,
        # 258 GB of memory and 48, later 96, slots (the other 62 hosts: 193 GB, 160 slots), and
        # 286 to 374 W (the others idle at about 390 W): another configuration, left out of the
        # model. qh2-rcc264 also never reports power.
        "exclude": ["qh2-rcc263", "qh2-rcc264"],
    },
    "workloads": {
        "slot_seconds": 300,
        # UFCG reports percent of one vCPU, not MHz. One vCPU at 100% is taken as f_ref MHz
        # so that Eq. (14), (15), (39) keep their MHz units.
        "f_ref_mhz": 2400.0,
        "drop_flavors": ["amphora"],    # load-balancer appliances, not user workloads
        "min_flavor_ram_mb": 1024,      # removes the flavor with ram = 4
        "min_snapshots": 3,
        "min_usage_coverage": 0.90,
        "max_fill_slots": 3,            # longer gaps are left out of the work window T^W_i
        "edge_slots": 2,                # alive this close to trace start/end = censored
        "utc_offset_hours": -3,         # Campina Grande; used for time of day only
        # Some hosts never report usage, so their VMs have no work content. False: those
        # VMs are dropped. True: each gets the mean usage of a measured VM of the same
        # flavor, drawn from the `impute_donor_pool` nearest in lifetime, and is flagged
        # `usage_imputed`. On: without these VMs the catalogue fills only about half of the
        # memory of ten servers, short of the normal and heavy days.
        "impute_unmonitored": True,
        "impute_donor_pool": 5,
        "impute_seed": 42,
    },
}

# Part 3 only: how the cleaned tables are turned into model inputs. These are modelling
# assumptions, not cleaning decisions.
#
# `servers10` holds the reference settings: the data center the thesis describes, ten
# servers on a normal day: three Dell R840 at 2.1 GHz, four R840 at 2.4 GHz and three Dell
# C6320 at 2.6 GHz, the C6320s with cold plates (and CRAC air, like every server). Every
# scenario is built from it, mostly with some values changed, recorded in its params.json:
#   servers10       the reference settings as they are
#   servers6        six servers: two R840 at 2.1 GHz, two R840 at 2.4 GHz, two C6320 at 2.6 GHz,
#                   on a normal day
#   servers10_light, servers10_heavy, servers6_light, servers6_heavy
#                   the same fleet on a light or heavy day: workloads drawn until peak
#                   demand reaches 30 % or 85 % of the fleet's cores or memory (normal: 60 %)
#   <scenario>_swing
#                   the same workloads on a grid with a deeper daily carbon swing
#                   (amplitude 0.50, not 0.25)
#   small           the same construction at a size suited to debugging a solver
servers10 = {
    "seed": 42,
    "dt_seconds": 300,
    "periods": 288,                     # 24 h
    "start_hour_local": 6,              # period 1 = 06:00, so the daytime arrival peak and
                                        # its deadlines both fall inside the horizon
    # One row per server, in server order; the number of servers and of zones follow from it.
    # Each row's host is drawn by seed from the eligible hosts of its model. The frequencies
    # are assumptions taken from SPEC CPU results for these models: 2.1 GHz Xeon Gold 6130 and
    # 2.4 GHz Xeon Gold 6148 (R840), 2.6 GHz Xeon E5-2690 v4 (C6320); the datasets name no
    # processor. Zones alternate 1, 2, ...; cold_plate = 1 puts a cold plate on the server.
    "fleet": [
        {"model": "R840", "f_MHz": 2100, "zone": 1, "cold_plate": 0},
        {"model": "R840", "f_MHz": 2100, "zone": 2, "cold_plate": 0},
        {"model": "R840", "f_MHz": 2100, "zone": 1, "cold_plate": 0},
        {"model": "R840", "f_MHz": 2400, "zone": 2, "cold_plate": 0},
        {"model": "R840", "f_MHz": 2400, "zone": 1, "cold_plate": 0},
        {"model": "R840", "f_MHz": 2400, "zone": 2, "cold_plate": 0},
        {"model": "R840", "f_MHz": 2400, "zone": 1, "cold_plate": 0},
        {"model": "C6320", "f_MHz": 2600, "zone": 2, "cold_plate": 1},
        {"model": "C6320", "f_MHz": 2600, "zone": 1, "cold_plate": 1},
        {"model": "C6320", "f_MHz": 2600, "zone": 2, "cold_plate": 1},
    ],
    # Workloads are drawn from the pooled catalogue (keeping time of day) until the peak
    # requested vCPUs reach this share of fleet cores, or the peak requested memory this
    # share of fleet memory, whichever comes first (or the catalogue runs out). Almost every
    # workload asks for 2 GB per vCPU, so memory fills first: a 64 GB R840 is full at
    # 32 vCPUs.
    "target_peak_load": 0.60,
    # If the whole catalogue does not reach the target (ten servers on a heavy day), it is
    # drawn again in the same order: each copy keeps its arrival time and gets "-r1", ...
    # appended to its ID. False: the scenario stops short of the target.
    "top_up_by_repeats": True,
    "min_et_periods": 3,                # under 15 core-minutes per vCPU in a whole life: idle VMs
    "max_et_periods": 144,              # longest baseline execution time admitted (12 h)
    "rt_share": 0.30,                   # share flagged time-sensitive (RT_i = 1)
    "rt_slack_frac": 0.25,              # RT slack = max(1, ceil(frac x ET_base)) periods
    "dt_slack_mult": 2.0,               # delay-tolerant slack = mult x ET_base, clamped to
    "dt_slack_min": 24,                 #   [2 h,
    "dt_slack_max": 96,                 #    8 h] so work can move across the carbon profile
    # Carbon intensity: Philippines 2024 annual mean (Our World in Data) with a SYNTHETIC
    # daily shape, because no hourly series is available.
    "ci_mean_g_per_kwh": 621.73,
    "ci_amplitude": 0.25,               # 0 gives the flat annual value
}
# The fleets and the three load levels (peak demand as a share of the fleet): ten servers,
# the reference settings as they are, and six. Other fleets go in the same dict, e.g. two
# 2.4 GHz R840s {"fleet": [{"model": "R840", "f_MHz": 2400, "zone": 1, "cold_plate": 0},
#                          {"model": "R840", "f_MHz": 2400, "zone": 2, "cold_plate": 0}]}.
fleets = {
    "servers10": {},
    "servers6": {"fleet": [
        {"model": "R840", "f_MHz": 2100, "zone": 1, "cold_plate": 0},
        {"model": "R840", "f_MHz": 2100, "zone": 2, "cold_plate": 0},
        {"model": "R840", "f_MHz": 2400, "zone": 1, "cold_plate": 0},
        {"model": "R840", "f_MHz": 2400, "zone": 2, "cold_plate": 0},
        {"model": "C6320", "f_MHz": 2600, "zone": 1, "cold_plate": 1},
        {"model": "C6320", "f_MHz": 2600, "zone": 2, "cold_plate": 1},
    ]},
}
levels = {"light": 0.30, "normal": 0.60, "heavy": 0.85}
cfg["scenario"] = {}
for fleet, change in fleets.items():
    for level, target in levels.items():
        name = fleet if level == "normal" else f"{fleet}_{level}"
        sc = servers10 | change | {"target_peak_load": target}
        cfg["scenario"][name] = sc
        cfg["scenario"][f"{name}_swing"] = sc | {"ci_amplitude": 0.50}
cfg["scenario"] |= {
    "small": servers10 | {"periods": 96, "start_hour_local": 12,   # 8 h, 12:00-20:00: spans the
                          "fleet": [                                # carbon dip and peak
                              {"model": "R840", "f_MHz": 2100, "zone": 1, "cold_plate": 0},
                              {"model": "R840", "f_MHz": 2400, "zone": 2, "cold_plate": 0},
                              {"model": "C6320", "f_MHz": 2600, "zone": 1, "cold_plate": 1},
                          ],
                          "max_et_periods": 36, "dt_slack_min": 12, "dt_slack_max": 36},
}
# Scenarios made from another scenario rather than from the cleaned tables: `small` with some
# workloads already running when the horizon starts (B_i = 1, Eq. 29) and one server down for
# maintenance (AV = 0, Eq. 11). Neither equation binds in any other scenario.
cfg["running_variants"] = {
    "small_running": {
        "from": "small",
        "running_arrivals": [3, 5],                       # B_i = 1 for workloads arriving in periods 3-5
        "outage": {"server": 3, "periods": [37, 48]},     # server 3 down 15:00-16:00
    },
}
# Constants changed for one scenario only; every other value comes from cfg["constants"].
# For example {"servers6": {"T_crac_max_C": 32.0}} for a wider cooling envelope.
cfg["scenario_constants"] = {}

# Model constants as printed in the thesis. They are passed through to each scenario's
# params.json; cleaning does not depend on them.
cfg["constants"] = {
    "R_dvfs": 0.378,
    "F_throttle": 0.622,  # printed 0.662; 1 - R_dvfs (typo, decided)
    "T_max_server_C": 90.0,
    "H_min_C": 10.0,
    "c_pw": 4182.0,
    "c_pa": 1.00643,
    "T_dvfs_C": 75.0,
    "T_throttle_C": 100.0,
    "Q_crac_kW": 116.0,
    "COP_crac_coeffs": [0.0068, 0.0008, 0.458],
    "T_crac_min_C": 18.0,
    "T_crac_max_C": 27.0,
    "FR_crac_min_m3s": 0.0,
    "FR_crac_max_m3s": 6.56,
    "Q_cp_kW": 0.3,
    "T_cp_min_K": 293.15,
    "T_cp_max_K": 313.1,
    "FR_cp_min_Lmin": 0.05,           # printed 0.005; the thesis group confirmed 0.05
    "FR_cp_max_Lmin": 0.35,
    "P_cp_rated_as_printed_mW": 0.657,
    "FR_cp_rated_Lmin": 0.264,
    "T_max_zone_C": 32.0,
}

# Cold-plate model agreed with the thesis group (IE_Thesis_Master_Document.pdf, 2026-10-14),
# passed through to each scenario's params.json. Flow is chosen from these settings;
# FR_cp_min/max are only the limits of Eq. (47), FR_cp_rated only the pump reference of Eq. (48).
cfg["cold_plate"] = {
    "flow_settings_Lmin": [0.10, 0.15, 0.20, 0.25, 0.30],
    "R_C_per_W": [0.1543, 0.1143, 0.0943, 0.0843, 0.0793],
    "water_density_kg_per_L": 1.0,
    "source": "Thesis group, IE_Thesis_Master_Document.pdf (2026-10-14): R = (T_cpu,max - T_in) / P_cpu "
              "from Wang et al. (2026), Fig. 10(a), CPU 1, at 200 W. Flow is chosen from these settings; "
              "0.05 and 0.35 L/min are only the limits of Eq. (47); 0.264 L/min is only the pump "
              "reference flow of Eq. (48) (0.657 mW, Wang and Tao 2025).",
}

In [2]:
import csv
import importlib.util
import json
import math
import random
import subprocess

if importlib.util.find_spec("duckdb") is None:      # e.g. on Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)

import duckdb
import numpy as np

RAW = [ROOT / cfg["paths"]["unimelb"], ROOT / cfg["paths"]["unimelb_c6320"]] + [
    ROOT / cfg["paths"]["ufcg"] / f for f in ("servers_usage.csv", "servers_specs.csv", "flavors.csv")]
missing = [str(p) for p in RAW if not p.exists()]
if missing:
    raise FileNotFoundError("raw data not found; see 'Before running' at the top:\n  " + "\n  ".join(missing))
# Sizes of the copy the reference scenarios were built from. Another size means another copy
# of the data: the notebook still runs, and section 4 shows whether the scenarios differ.
RAW_BYTES = {"UniMelb.csv": 613_212_066, "UniMelb_C6320.csv": 184_178_675, "servers_usage.csv": 1_938_998_192,
             "servers_specs.csv": 1_497_948_679, "flavors.csv": 1_496}
for p in RAW:
    if p.stat().st_size != RAW_BYTES[p.name]:
        print(f"Note: {p.name} is {p.stat().st_size:,} bytes; the reference copy is {RAW_BYTES[p.name]:,}.")
CLEAN = ROOT / cfg["paths"]["clean"]
CLEAN.mkdir(parents=True, exist_ok=True)


def connect():
    con = duckdb.connect()
    con.execute("SET threads=8; SET memory_limit='9GB'; SET preserve_insertion_order=false")
    return con


def scalar(con, sql):
    return con.execute(sql).fetchone()[0]


def rows(con, sql):
    cur = con.execute(sql)
    cols = [d[0] for d in cur.description]
    return [dict(zip(cols, r)) for r in cur.fetchall()]


class Log:
    # Records what each cleaning step removed, so every count can be reproduced.
    def __init__(self, dataset, unit):
        self.dataset, self.unit, self.entries = dataset, unit, []

    def step(self, name, before, after, note=""):
        self.entries.append([self.dataset, name, self.unit, before, after, before - after, note])
        print(f"  {name:<44} {before:>10,} -> {after:>10,} {self.unit}  {note}")

    def write(self, path):
        with open(path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["dataset", "step", "unit", "before", "after", "removed", "note"])
            w.writerows(self.entries)


def write_csv(path, records, fields=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = fields or list(records[0].keys())
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(records)

In [3]:
# The release this notebook belongs to, and the fingerprint of each scenario it writes
# when the settings are left as shipped. Section 4 compares the scenarios written
# below with these. This cell is generated; don't edit it.
VERSION = '1.3.0'
REFERENCE_FINGERPRINTS = {
    'servers10': '3edf034fdf46',
    'servers10_heavy': '627e163ebe02',
    'servers10_heavy_swing': 'bfcae8a67536',
    'servers10_light': '25f23c3dfb79',
    'servers10_light_swing': 'dc1018a911c2',
    'servers10_swing': '81dbb921dc62',
    'servers6': '94352bda5aa7',
    'servers6_heavy': '9f9a249a2d1e',
    'servers6_heavy_swing': 'bde23dcd3ffb',
    'servers6_light': 'cb0c78e1602e',
    'servers6_light_swing': '9c7b078d7cd7',
    'servers6_swing': 'fdc4e53b0393',
    'small': 'ef6702bb949b',
    'small_running': '42b739688470',
}
print(f"ThermaSched {VERSION}")


ThermaSched 1.3.0


## 1. Servers (UniMelb)

UniMelb gives 10-minute telemetry for the physical servers of one cloud, in two datasets,
one per server model:

| Dataset | Server model | Hosts | Period | Sections |
|---|---|---|---|---|
| `UniMelb.csv` | Dell PowerEdge R840 | 81 (`qh2-rcc120` to `qh2-rcc200`) | Oct 2018 - Jun 2019 | 1.1 - 1.4 |
| `UniMelb_C6320.csv` | Dell PowerEdge C6320 | 64 (`qh2-rcc201` to `qh2-rcc264`) | Feb - Jun 2019 | 1.5 - 1.7 |

The model needs only static values per server: idle and maximum power (Eq. 17-19),
initial temperature (Eq. 30), core and memory capacity, and a thermal response. Because
no time series is carried into the model, the 10-minute records are not resampled to the
model's 5-minute period. Both models go through the same steps and the same gates; section
1.8 writes them to one table, with the model in a column.

### 1.1 R840: load the data and keep the 64-core period

`CPU_cores` is 64 for every server until 2019-03-14 and 128 from 2019-03-15, when
hyper-threads start being counted. Load is then reported on a different scale and idle
power shifts, so records from before and after the change cannot be pooled into one
power curve. Only the 64-core period is kept. The same cut also removes the day of empty
records between the two periods and the repeated daylight-saving hour of 2019-04-07.

Server names are trimmed of stray spaces, and the higher of the two CPU temperatures is
taken as the server temperature, because the hotter CPU is the one that reaches the
thermal limit first.

In [4]:
sc = cfg["servers"]
con_s = connect()
slog = Log("UniMelb R840", "rows")

con_s.execute(f'''
    CREATE TABLE raw AS
    SELECT trim(Series) AS server, Time::TIMESTAMP AS t, CPU_Load AS cpu, Power AS pw,
           Inlet_Temperature AS inlet, Fan_speed1 AS fan,
           greatest(CPU1_Temperature, CPU2_Temperature) AS temp,
           Ram AS ram, CPU_cores AS cores
    FROM read_csv('{ROOT / cfg["paths"]["unimelb"]}', header=true)''')
n0 = scalar(con_s, "SELECT count(*) FROM raw")

con_s.execute(f'''CREATE TABLE a AS SELECT * FROM raw
                  WHERE t < TIMESTAMP '{sc["regime_end"]}' AND cores = {sc["cores"]}''')
n1 = scalar(con_s, "SELECT count(*) FROM a")
slog.step("keep 64-core period", n0, n1, f"before {sc['regime_end']}")

  keep 64-core period                           2,748,525 ->  1,621,427 rows  before 2019-03-14 11:40:00


### 1.2 R840: duplicates, missing values, failed sensors

| Step | Why |
|---|---|
| keep one record per server and time | a repeated timestamp would be counted twice in every average and would break the record-to-record thermal fit |
| drop records missing `CPU_Load` or `Power` | every server parameter is built from these two columns |
| set failed-sensor inlet readings to missing | an inlet reading of 4.0 °C together with a fan speed of 280 rpm is a sensor fault, not a real temperature |

The faulty inlet values are set to missing, not dropped, so the load and power on the
same record are still used. The first two steps remove nothing in this dataset, because
the affected records all fall outside the 64-core period. They are kept as safeguards.

In [5]:
n2 = scalar(con_s, "SELECT count(*) FROM (SELECT DISTINCT server, t FROM a)")
con_s.execute("CREATE TABLE b AS SELECT * FROM a QUALIFY row_number() OVER (PARTITION BY server, t ORDER BY pw) = 1")
slog.step("drop duplicate (server, time)", n1, n2)

con_s.execute("CREATE TABLE c AS SELECT * FROM b WHERE cpu IS NOT NULL AND pw IS NOT NULL")
n3 = scalar(con_s, "SELECT count(*) FROM c")
slog.step("drop rows missing CPU_Load or Power", n2, n3)

bad_inlet = scalar(con_s, f"SELECT count(*) FROM c WHERE inlet = {sc['inlet_sentinel']} AND fan = {sc['fan_sentinel']}")
con_s.execute(f"UPDATE c SET inlet = NULL WHERE inlet = {sc['inlet_sentinel']} AND fan = {sc['fan_sentinel']}")
slog.step("null failed-sensor inlet readings", n3, n3, f"{bad_inlet:,} inlet values set to null")

  drop duplicate (server, time)                 1,621,427 ->  1,621,427 rows  
  drop rows missing CPU_Load or Power           1,621,427 ->  1,621,427 rows  
  null failed-sensor inlet readings             1,621,427 ->  1,621,427 rows  53,165 inlet values set to null


### 1.3 R840: parameters per server

| Parameter | Rule |
|---|---|
| $P^{idle}_s$ | Eq. (17)-(18): mean power where `CPU_Load` ≤ 1 (the load is on a 0-100 scale) |
| $P^{max}_s$ | Eq. (19): highest observed power |
| $T^0_s$ | Eq. (30): server temperature at the first kept record |
| $M_s$ | largest `Ram` × 1,024 MB: `Ram` is in GB, and each R840 host has 64 GB (Ilager et al., UCC 2023), so $M_s$ = 66,036 MB |

A linear fit of power on load, evaluated at 100% load, is stored as
`P_fit_full_load_W` for comparison with $P^{max}_s$.

Completeness is the share of 10-minute records a server has over the whole kept period,
not over its own first-to-last record. A server that joined late is therefore judged as
incomplete, not on a short record.

In [6]:
stats = rows(con_s, f'''
    WITH w AS (SELECT min(t) w0, max(t) w1 FROM c)
    SELECT server, '{sc["model"]}' AS model,
           count(*) AS n_rows,
           count(*) / (epoch((SELECT w1 - w0 FROM w)) / 600 + 1) AS completeness,
           max(cpu) AS peak_cpu_load,
           corr(cpu, pw) AS power_load_corr,
           count(*) FILTER (cpu <= {sc["idle_cpu_load"]}) AS n_idle_rows,
           avg(pw) FILTER (cpu <= {sc["idle_cpu_load"]}) AS P_idle_W,
           max(pw) AS P_max_W,
           regr_intercept(pw, cpu) + 100 * regr_slope(pw, cpu) AS P_fit_full_load_W,
           arg_min(temp, t) AS T0_C,
           max(temp) AS peak_temp_C,
           avg(inlet) AS mean_inlet_C, 'measured' AS inlet_source,
           {sc["cores"]} AS N_cores,
           max(ram) * {sc["ram_unit_gb"]} * 1024 AS M_s_MB
    FROM c GROUP BY server ORDER BY server''')
print(len(stats), "servers")

81 servers


### 1.4 R840: thermal response and eligibility

The model needs a relation between a server's power draw and its temperature. The
formulation does not give one, so a first-order response is fitted for each server from
pairs of consecutive 10-minute records:

- steady state: $T_{cpu} = T_{inlet} + a + R \cdot P$
- per period: $T_t = \alpha\,T_{t-1} + (1-\alpha)\cdot\text{steady state}$

The inlet coefficient is fixed at 1 because inlet temperature barely varies in the data,
which leaves too little signal to estimate it. The 5-minute $\alpha$ is the square root
of the 10-minute one, since two 5-minute steps make one 10-minute step.

A server is **eligible** for the model only if it passes every gate below. The others
stay in `servers.csv` with `eligible = 0`.

| Gate | Why |
|---|---|
| at least 90% complete | parameters should describe the whole period, not a fragment of it |
| reaches 30% CPU load | a server seen only near idle gives no evidence of its power at higher load |
| power-to-load correlation of at least 0.8 | the model treats power as rising with load (Eq. 20). A server whose power does not follow its load does not fit that |
| at least 30 near-idle records, and $P^{max}_s > P^{idle}_s$ | idle power is an average, so it needs enough records behind it |
| valid thermal fit | at least 500 consecutive pairs, $0 < \alpha < 1$ and $R > 0$, so that temperature rises with power and settles |

In [7]:
def thermal_fit(t, temp, power, inlet):
    # rise = T_cpu - T_inlet;  rise[k] = alpha*rise[k-1] + (1-alpha)*(a + R*P[k])
    rise = temp - inlet
    ok = (np.diff(t) == 600) & np.isfinite(rise[1:]) & np.isfinite(rise[:-1]) & np.isfinite(power[1:])
    if ok.sum() < 500:
        return None
    y = rise[1:][ok]
    X = np.column_stack([rise[:-1][ok], power[1:][ok], np.ones(ok.sum())])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    alpha = float(coef[0])
    if not 0.0 < alpha < 1.0:
        return None
    R, a = (coef[1:] / (1.0 - alpha)).tolist()
    rmse = float(np.sqrt(np.mean((y - (a + R * power[1:][ok])) ** 2)))
    return {"th_a_C": a, "th_R_C_per_W": R, "th_alpha_10min": alpha,
            "th_alpha_5min": alpha ** 0.5, "th_steady_rmse_C": rmse}


def fit_and_gate(con, table, stats, excluded=()):
    # Fits the thermal response of every server in `stats` from its records in `table`, and
    # sets `eligible`. Returns the failures per gate and the gates each server failed.
    gate_fail = {"completeness": 0, "peak load": 0, "correlation": 0, "idle records": 0, "thermal fit": 0}
    gate_fail |= {"configuration": 0} if excluded else {}
    failed = {}
    for s in stats:
        t, temp, pw, inlet = con.execute(
            f"SELECT epoch(t), temp, pw, inlet FROM {table} WHERE server = ? ORDER BY t", [s["server"]]
        ).fetchnumpy().values()
        fit = thermal_fit(np.asarray(t, float), *(np.ma.filled(np.ma.asarray(x, float), np.nan) for x in (temp, pw, inlet)))
        s.update(fit or dict.fromkeys(
            ["th_a_C", "th_R_C_per_W", "th_alpha_10min", "th_alpha_5min", "th_steady_rmse_C"]))
        s["C_s_cores"] = s["N_cores"]
        gates = {
            "completeness": s["completeness"] >= sc["min_completeness"],
            "peak load": (s["peak_cpu_load"] or 0) >= sc["min_peak_load"],
            "correlation": (s["power_load_corr"] or 0) >= sc["min_power_load_corr"],
            "idle records": s["n_idle_rows"] >= sc["min_idle_rows"] and s["P_max_W"] > s["P_idle_W"],
            "thermal fit": fit is not None and fit["th_R_C_per_W"] > 0,
        }
        gates |= {"configuration": s["server"] not in excluded} if excluded else {}
        for g, passed in gates.items():
            gate_fail[g] += not passed
        failed[s["server"]] = [g for g, passed in gates.items() if not passed]
        s["eligible"] = int(all(gates.values()))
        for k, v in s.items():
            if isinstance(v, float):
                s[k] = round(v, 5)
    return gate_fail, failed


gate_fail, _ = fit_and_gate(con_s, "c", stats)
srv_log = Log("UniMelb R840", "servers")
srv_log.step("servers passing power-curve and thermal gates", len(stats), sum(s["eligible"] for s in stats))
slog.entries += srv_log.entries
print("failures per gate (a server can fail several):", gate_fail)


  servers passing power-curve and thermal gates         81 ->         45 servers  
failures per gate (a server can fail several): {'completeness': 13, 'peak load': 25, 'correlation': 16, 'idle records': 2, 'thermal fit': 6}


### 1.5 C6320: load and clean

The C6320 dataset has the R840's columns except `Inlet_Temperature` and the fan speeds, and
covers 1 February to 20 June 2019. It is cleaned the same way as the R840 data, with two
differences:

| Step | Why |
|---|---|
| drop records at a doubled slot count (above 160) | `CPU_cores` counts vCPU slots. The cloud doubled them from 160 to 320 on some C6320 hosts from 15 March 2019, each host on its own date, so the cut is made per record, not at one instant. This keeps one power curve per host, as the R840's 64-core cut does. About half of the records up to 15 March carry no count at all; no host had doubled yet, so they are kept |
| keep one record per server and time | as for the R840. Here it removes the repeated daylight-saving hour of 2019-04-07, which the R840's 64-core cut removed before |
| drop records missing `CPU_Load` or `Power` | as for the R840 |

No failed-sensor step is needed, since there is no inlet or fan reading to check.

In [8]:
sc6 = cfg["servers_c6320"]
clog = Log("UniMelb C6320", "rows")

con_s.execute(f'''
    CREATE TABLE raw2 AS
    SELECT trim(Series) AS server, Time::TIMESTAMP AS t, CPU_Load AS cpu, Power AS pw,
           greatest(CPU1_Temperature, CPU2_Temperature) AS temp,
           Ram AS ram, CPU_cores AS cores
    FROM read_csv('{ROOT / cfg["paths"]["unimelb_c6320"]}', header=true)''')
m0 = scalar(con_s, "SELECT count(*) FROM raw2")
print(scalar(con_s, "SELECT count(DISTINCT server) FROM raw2"), "servers,",
      *con_s.execute("SELECT min(t), max(t) FROM raw2").fetchone())

con_s.execute(f"CREATE TABLE a2 AS SELECT * FROM raw2 WHERE cores IS NULL OR cores <= {sc6['max_slots']}")
m1 = scalar(con_s, "SELECT count(*) FROM a2")
clog.step("drop records at a doubled slot count", m0, m1, f"CPU_cores above {sc6['max_slots']}")

m2 = scalar(con_s, "SELECT count(*) FROM (SELECT DISTINCT server, t FROM a2)")
con_s.execute("CREATE TABLE b2 AS SELECT * FROM a2 QUALIFY row_number() OVER (PARTITION BY server, t ORDER BY pw) = 1")
clog.step("drop duplicate (server, time)", m1, m2)

con_s.execute("CREATE TABLE c2 AS SELECT * FROM b2 WHERE cpu IS NOT NULL AND pw IS NOT NULL")
m3 = scalar(con_s, "SELECT count(*) FROM c2")
clog.step("drop rows missing CPU_Load or Power", m2, m3)


64 servers, 2019-02-01 11:10:00 2019-06-20 10:00:00


  drop records at a doubled slot count          1,281,024 ->  1,179,631 rows  CPU_cores above 160
  drop duplicate (server, time)                 1,179,631 ->  1,179,283 rows  
  drop rows missing CPU_Load or Power           1,179,283 ->  1,030,643 rows  


### 1.6 C6320: inlet temperature from the R840 room

The thermal fit of section 1.4 needs the inlet temperature, and the C6320 dataset has none.
The gap follows when a host joined the monitoring, not its model: the R840 hosts that
report inlet and fans have records from October 2018, while every host whose records start
on 1 February 2019 reports neither, the 64 C6320s and also five R840s (`qh2-rcc196` to
`qh2-rcc200`, listed below).

**Assumption.** The C6320s are taken to draw the same room air as the R840s. Their inlet at
each 10-minute timestamp is the mean measured inlet of the R840 hosts with a valid reading
at that timestamp (the failed-sensor value of section 1.2 excluded). The R840 inlet varies
little (in 1 °C steps, and neighbouring hosts often read the same value), so the room
mean is a stable series. C6320 records at a timestamp with no R840 reading have no inlet
and are left out of the thermal fit. `inlet_source` in `servers.csv` records which inlet a
server's fit used: `measured` or `d1_room_mean`.

In [9]:
print("R840 hosts with no inlet reading at all:")
print(con_s.sql('''SELECT server, min(t) AS first_record, count(*) AS records, count(inlet) AS inlet_readings,
                          count(fan) AS fan_readings
                   FROM raw GROUP BY server HAVING count(inlet) = 0 ORDER BY server'''))
print("first C6320 record:", scalar(con_s, "SELECT min(t) FROM raw2"))

# Room inlet: every R840 record (both core-count periods), one per server and time, failed-sensor
# readings left out, averaged over the hosts at each timestamp.
con_s.execute(f'''
    CREATE TABLE room AS
    SELECT t, avg(inlet) AS inlet, count(*) AS n_hosts
    FROM (SELECT * FROM raw QUALIFY row_number() OVER (PARTITION BY server, t ORDER BY pw) = 1)
    WHERE inlet IS NOT NULL AND NOT (inlet = {sc['inlet_sentinel']} AND fan = {sc['fan_sentinel']})
    GROUP BY t''')
con_s.execute("CREATE TABLE c2i AS SELECT c2.*, room.inlet FROM c2 LEFT JOIN room USING (t)")
n, mean, sd, lo, hi, hosts = con_s.execute('''
    SELECT count(*), avg(inlet), stddev_samp(inlet), min(inlet), max(inlet), median(n_hosts) FROM room
    WHERE t BETWEEN (SELECT min(t) FROM raw2) AND (SELECT max(t) FROM raw2)''').fetchone()
print(f"R840 room inlet over the C6320 period: {n:,} timestamps, mean {mean:.2f} C, SD {sd:.2f} C, "
      f"range {lo:.2f} to {hi:.2f} C, median {hosts:.0f} R840 hosts per timestamp")
no_inlet = scalar(con_s, "SELECT count(*) FROM c2i WHERE inlet IS NULL")
print(f"C6320 records with no R840 reading at their timestamp: {no_inlet:,} of {m3:,}")


R840 hosts with no inlet reading at all:
┌────────────┬─────────────────────┬─────────┬────────────────┬──────────────┐
│   server   │    first_record     │ records │ inlet_readings │ fan_readings │
│  varchar   │      timestamp      │  int64  │     int64      │    int64     │
├────────────┼─────────────────────┼─────────┼────────────────┼──────────────┤
│ qh2-rcc196 │ 2019-02-01 11:10:00 │   16747 │              0 │            0 │
│ qh2-rcc197 │ 2019-02-01 11:10:00 │   16748 │              0 │            0 │
│ qh2-rcc198 │ 2019-02-01 11:10:00 │   16748 │              0 │            0 │
│ qh2-rcc199 │ 2019-02-01 11:10:00 │   16735 │              0 │            0 │
│ qh2-rcc200 │ 2019-02-01 11:10:00 │   16734 │              0 │            0 │
└────────────┴─────────────────────┴─────────┴────────────────┴──────────────┘

first C6320 record: 2019-02-01 11:10:00


R840 room inlet over the C6320 period: 19,882 timestamps, mean 18.84 C, SD 0.62 C, range 17.57 to 22.62 C, median 71 R840 hosts per timestamp
C6320 records with no R840 reading at their timestamp: 7,454 of 1,030,643


### 1.7 C6320: parameters and eligibility

The rules of sections 1.3 and 1.4 apply unchanged, including every gate, with these
differences:

| Item | C6320 | Why |
|---|---|---|
| completeness | over the C6320 dataset's own period, 1 February to 20 June 2019 | the R840 period does not apply to a dataset that starts later |
| $N^{cores}_s$, $C_s$ | 80 | `CPU_cores` counts the cloud's vCPU slots. The first C6320 hosts show 80 in their first three days and 160 afterwards, the cloud allowing two vCPUs per hardware thread. 80 is the count before that doubling, as 64 is for the R840 |
| $M_s$ | median of the host's `Ram` × 1,024 MB = 197,972 MB (193 GB) | `Ram` varies in its last decimals on these hosts, so the median of all the host's records is taken |
| inlet | the R840 room mean of section 1.6 | the dataset has no inlet |
| configuration gate | `qh2-rcc263` and `qh2-rcc264` fail it | they have 258 GB of memory and 48, later 96, slots, where the other 62 hosts have 193 GB and 160; `qh2-rcc263` draws 286 to 374 W where the others idle at about 390 W, and `qh2-rcc264` never reports power. A different configuration from the rest, so they are kept in the table but not eligible |

In [10]:
con_s.execute(f'''
    CREATE TABLE stats2 AS
    WITH w AS (SELECT min(t) w0, max(t) w1 FROM c2i),
         mem AS (SELECT server, median(ram) AS ram_gb FROM raw2 GROUP BY server)
    SELECT server, '{sc6["model"]}' AS model,
           count(t) AS n_rows,
           count(t) / (epoch((SELECT w1 - w0 FROM w)) / 600 + 1) AS completeness,
           max(cpu) AS peak_cpu_load,
           corr(cpu, pw) AS power_load_corr,
           count(t) FILTER (cpu <= {sc["idle_cpu_load"]}) AS n_idle_rows,
           avg(pw) FILTER (cpu <= {sc["idle_cpu_load"]}) AS P_idle_W,
           max(pw) AS P_max_W,
           regr_intercept(pw, cpu) + 100 * regr_slope(pw, cpu) AS P_fit_full_load_W,
           arg_min(temp, t) AS T0_C,
           max(temp) AS peak_temp_C,
           avg(inlet) AS mean_inlet_C, 'd1_room_mean' AS inlet_source,
           {sc6["cores"]} AS N_cores,
           any_value(ram_gb) * {sc["ram_unit_gb"]} * 1024 AS M_s_MB
    FROM mem LEFT JOIN c2i USING (server)      -- every host, also one with no record left
    GROUP BY server ORDER BY server''')
stats2 = rows(con_s, "SELECT * FROM stats2 ORDER BY server")
gate_fail2, failed2 = fit_and_gate(con_s, "c2i", stats2, excluded=set(sc6["exclude"]))
srv_log2 = Log("UniMelb C6320", "servers")
srv_log2.step("servers passing power-curve and thermal gates", len(stats2), sum(s["eligible"] for s in stats2))
clog.entries += srv_log2.entries
print("failures per gate (a server can fail several):", gate_fail2)
print("not eligible:")
for server, gates in failed2.items():
    if gates:
        print(f"  {server}: {', '.join(gates)}")


  servers passing power-curve and thermal gates         64 ->         44 servers  
failures per gate (a server can fail several): {'completeness': 19, 'peak load': 6, 'correlation': 3, 'idle records': 4, 'thermal fit': 2, 'configuration': 2}
not eligible:
  qh2-rcc201: completeness
  qh2-rcc202: completeness
  qh2-rcc203: completeness
  qh2-rcc204: completeness, idle records
  qh2-rcc205: completeness, peak load
  qh2-rcc207: completeness, correlation
  qh2-rcc208: completeness
  qh2-rcc226: completeness, peak load, correlation, idle records
  qh2-rcc240: completeness
  qh2-rcc241: completeness
  qh2-rcc242: completeness
  qh2-rcc244: completeness, peak load
  qh2-rcc247: completeness
  qh2-rcc250: completeness
  qh2-rcc256: completeness
  qh2-rcc257: peak load
  qh2-rcc258: completeness
  qh2-rcc262: completeness
  qh2-rcc263: completeness, peak load, idle records, thermal fit, configuration
  qh2-rcc264: completeness, peak load, correlation, idle records, thermal fit, configuration


### 1.8 Both models in one table

The two tables are written as one, `servers.csv`, with the model in the `model` column.
Host names never repeat across the two datasets. The ranges below cover the eligible
servers of each model. A C6320 draws about two and a half times the power of an R840 at
idle. Few servers of either model pass 60% load, so `P_fit_full_load_W` extrapolates well
past the observed range.

In [11]:
servers = stats + stats2
write_csv(CLEAN / "servers.csv", servers)
slog.entries += clog.entries
slog.write(CLEAN / "cleaning_log_servers.csv")
con_s.close()

# Range of each parameter across the eligible servers of each model
n_el = {m: sum(s["eligible"] for s in servers if s["model"] == m) for m in ("R840", "C6320")}
print(f"{'':<20}" + "".join(f"{f'{m}, {n} eligible':>28}" for m, n in n_el.items()))
print(f"{'':<20}" + f"{'min':>10}{'median':>9}{'max':>9}" * 2)
for k in ["P_idle_W", "P_max_W", "P_fit_full_load_W", "T0_C", "mean_inlet_C", "th_R_C_per_W", "th_a_C",
          "th_alpha_5min", "th_steady_rmse_C"]:
    line = f"{k:<20}"
    for m in ("R840", "C6320"):
        v = sorted(s[k] for s in servers if s["eligible"] and s["model"] == m)
        line += f"{v[0]:>10.3f}{v[len(v) // 2]:>9.3f}{v[-1]:>9.3f}" if v else f"{'-':>28}"
    print(line)


                               R840, 45 eligible          C6320, 44 eligible
                           min   median      max       min   median      max
P_idle_W                90.889  155.208  171.738   286.000  390.759  392.438
P_max_W                288.133  358.800  393.200   390.000  767.433  806.000
P_fit_full_load_W      454.399  588.721  767.088   393.473 1022.853 1350.854
T0_C                    32.067   62.983   76.617    36.000   53.817   60.533
mean_inlet_C            15.733   18.209   20.726    18.806   18.867   18.875
th_R_C_per_W             0.115    0.148    0.219     0.030    0.079    0.109
th_a_C                 -10.070    8.519   20.253   -18.422    0.865   19.414
th_alpha_5min            0.659    0.924    0.980     0.907    0.964    0.995
th_steady_rmse_C         1.477    2.557    4.886     2.036    3.319    4.670


## 2. Workloads (UFCG)

UFCG is an OpenStack trace. An OpenStack "server" is a virtual machine, so a UFCG
`server_id` is a **workload** $i$ in the model, not a server $s$. A workload is defined
as one VM whose whole life lies inside the trace.

### 2.1 Load usage on a 5-minute grid

Usage is collected every 5 minutes, but the snapshots are stamped a few seconds apart,
both from one snapshot to the next and between the usage and specification files. Raw
timestamps therefore do not line up. Every record is assigned to its nearest 5-minute
slot, which is also the model's period. A VM seen twice in one slot is averaged, so each
VM has at most one reading per slot.

Usage reported as `NA` is kept as missing here. Section 2.2 looks at where it comes
from before anything is removed.

In [12]:
wc, src = cfg["workloads"], ROOT / cfg["paths"]["ufcg"]
slot, f_ref = wc["slot_seconds"], wc["f_ref_mhz"]
con = connect()
rlog = Log("UFCG", "rows")

con.execute(f'''
    CREATE TABLE u AS
    SELECT server_id AS vm, CAST(round(timestamp / {slot}.0) AS BIGINT) AS slot,
           avg(vcpu_usage) AS cpu, any_value(host_id) AS host, count(*) AS n_raw
    FROM read_csv('{src / "servers_usage.csv"}', header=true, nullstr='NA',
         columns={{'id':'BIGINT','timestamp':'BIGINT','server_id':'VARCHAR',
                   'vcpu_usage':'DOUBLE','ram_usage':'DOUBLE','host_id':'VARCHAR'}})
    GROUP BY 1, 2''')
n_raw, n_slot = con.execute("SELECT sum(n_raw), count(*) FROM u").fetchone()
rlog.step("snap to 5-minute slots, merge doubles", n_raw, n_slot)
rlog.step("usage reported as NA", n_slot, scalar(con, "SELECT count(cpu) FROM u"), "kept as missing")

  snap to 5-minute slots, merge doubles        15,397,129 -> 15,395,669 rows  
  usage reported as NA                         15,395,669 -> 11,942,874 rows  kept as missing


### 2.2 Where the missing usage comes from

About 22% of usage rows are `NA`. How to treat them depends on whether the gaps are
scattered or concentrated, so the table below groups hosts by their share of `NA` rows.

The missing data is **host-level**. A group of hosts never reports CPU or RAM usage at
all. On every other host `NA` is rare: about 0.4% at the median, and under 3% on every
host with more than a few rows. A VM on a never-reporting host has no usage reading at
any point, so its work content is *unknown*, not zero. Treating `NA` as zero would turn
those VMs into workloads that need no work. They are removed in section 2.3, or
optionally given borrowed usage in section 2.6.

In [13]:
con.execute("CREATE TABLE dead AS SELECT host FROM u WHERE host IS NOT NULL GROUP BY host HAVING count(cpu) = 0")
n_dead, n_host = scalar(con, "SELECT count(*) FROM dead"), scalar(con, "SELECT count(DISTINCT host) FROM u")
print(f"{n_dead} of {n_host} hosts never report usage")
con.sql('''
    SELECT CASE WHEN na = 1 THEN 'never reports (100% NA)' WHEN na >= 0.05 THEN '5-99% NA'
                WHEN na >= 0.01 THEN '1-5% NA' ELSE 'under 1% NA' END AS host_group,
           count(*) AS hosts, sum(n) AS usage_rows, sum(n_na) AS na_rows
    FROM (SELECT host, count(*) AS n, count(*) - count(cpu) AS n_na, 1 - count(cpu) / count(*) AS na
          FROM u WHERE host IS NOT NULL GROUP BY host)
    GROUP BY 1 ORDER BY min(na) DESC''')

26 of 102 hosts never report usage


┌─────────────────────────┬───────┬────────────┬─────────┐
│       host_group        │ hosts │ usage_rows │ na_rows │
│         varchar         │ int64 │   int128   │ int128  │
├─────────────────────────┼───────┼────────────┼─────────┤
│ never reports (100% NA) │    26 │    3137748 │ 3137748 │
│ 5-99% NA                │     1 │          3 │       1 │
│ 1-5% NA                 │     2 │     399381 │    5415 │
│ under 1% NA             │    73 │   11598771 │   49865 │
└─────────────────────────┴───────┴────────────┴─────────┘

### 2.3 One row per VM, then the filters

Each VM is reduced to one row: first and last slot seen, number of readings, hosts it
ran on, and its flavor (the OpenStack size class that fixes its vCPUs and memory). The
filters are then applied in this order.

| Filter | Why |
|---|---|
| has a flavor record | requested cores $C^{req}_i$ and memory $M^{req}_i$ come from the flavor |
| drop `amphora` | these are load-balancer appliances run by the platform, not user workloads |
| drop flavors with under 1 GB of memory | one flavor lists `ram` = 4, which is a data-entry error |
| drop resized VMs | $C^{req}_i$ must be one value for the whole workload |
| drop migrated VMs | the model never moves a running workload, so a VM that ran on two hosts is not one workload. A host of `NA` is not counted as a second host |
| drop VMs alive at trace start | their true arrival time is before the trace and is unknown |
| drop VMs alive at trace end | they were still running, so their total work is unknown |
| at least 3 snapshots | fewer readings are too few to describe a VM's usage |
| drop VMs on never-reporting hosts | no usage at all (section 2.2). Skipped when imputation is switched on |
| has any usage reading | removes the few remaining VMs with no reading at all, whose host is always `NA` |

"Alive at trace start or end" means seen within 2 slots of the first or last slot of
the trace.

In [14]:
con.execute(f'''
    CREATE TABLE spec AS
    SELECT server_id AS vm, count(DISTINCT flavor_id) AS n_flavors, any_value(flavor_id) AS flavor_id
    FROM read_csv('{src / "servers_specs.csv"}', header=true,
         columns={{'id':'BIGINT','timestamp':'BIGINT','server_id':'VARCHAR','flavor_id':'VARCHAR'}})
    GROUP BY 1''')
con.execute(f"CREATE TABLE fl AS SELECT * FROM read_csv('{src / 'flavors.csv'}', header=true)")
con.execute("CREATE TABLE gs AS SELECT DISTINCT slot FROM u")   # slots the collector was up
g0, g1 = con.execute("SELECT min(slot), max(slot) FROM gs").fetchone()

con.execute('''
    CREATE TABLE vm AS
    SELECT v.*, s.n_flavors, f.flavor_name, f.vcpu, f.ram AS ram_mb
    FROM (SELECT u.vm, min(u.slot) AS s0, max(u.slot) AS s1, count(*) AS n_slots, count(u.cpu) AS n_cpu,
                 count(DISTINCT u.host) AS n_hosts, count(d.host) > 0 AS unmonitored
          FROM u LEFT JOIN dead d ON u.host = d.host GROUP BY u.vm) v
    LEFT JOIN spec s USING (vm) LEFT JOIN fl f ON s.flavor_id = f.flavor_id''')

log = Log("UFCG", "VMs")
drop = ", ".join(f"'{x}'" for x in wc["drop_flavors"])
impute = wc["impute_unmonitored"]
filters = [
    ("has a flavor record", "flavor_name IS NOT NULL", ""),
    ("drop infrastructure flavors", f"flavor_name NOT IN ({drop})", drop),
    ("drop implausible flavor memory", f"ram_mb >= {wc['min_flavor_ram_mb']}", ""),
    ("drop resized VMs", "n_flavors = 1", "C_req must be constant"),
    ("drop migrated VMs", "n_hosts <= 1", "host is NA for some VMs; that is not a migration"),
    ("drop VMs alive at trace start", f"s0 > {g0 + wc['edge_slots']}", "arrival unknown"),
    ("drop VMs alive at trace end", f"s1 < {g1 - wc['edge_slots']}", "total work unknown"),
    (f"at least {wc['min_snapshots']} snapshots", f"n_slots >= {wc['min_snapshots']}", ""),
    ("drop VMs on unmonitored hosts", "true" if impute else "NOT unmonitored",
     f"{n_dead} of {n_host} hosts never report usage" + ("; kept for imputation" if impute else "")),
    ("has any usage reading", "n_cpu > 0 OR unmonitored", ""),
]
con.execute("CREATE TABLE keep AS SELECT * FROM vm")
for name, cond, note in filters:
    before = scalar(con, "SELECT count(*) FROM keep")
    con.execute(f"DELETE FROM keep WHERE NOT coalesce({cond}, false)")
    log.step(name, before, scalar(con, "SELECT count(*) FROM keep"), note)

  has a flavor record                               2,800 ->      2,799 VMs  
  drop infrastructure flavors                       2,799 ->      1,193 VMs  'amphora'
  drop implausible flavor memory                    1,193 ->      1,191 VMs  
  drop resized VMs                                  1,191 ->      1,172 VMs  C_req must be constant
  drop migrated VMs                                 1,172 ->      1,166 VMs  host is NA for some VMs; that is not a migration
  drop VMs alive at trace start                     1,166 ->      1,054 VMs  arrival unknown
  drop VMs alive at trace end                       1,054 ->        944 VMs  total work unknown
  at least 3 snapshots                                944 ->        785 VMs  
  drop VMs on unmonitored hosts                       785 ->        785 VMs  26 of 102 hosts never report usage; kept for imputation
  has any usage reading                               785 ->        767 VMs  


### 2.4 Usage coverage

Each measured VM is laid out on a full grid, one row per slot from its first to its last
slot, so that gaps become visible as rows with no reading.

The collector itself was sometimes down, and in those slots no VM reports (the longest
such outage is 38.7 hours). The column `up` marks slots in which at least one VM
reported. Coverage is judged on those slots only, so an outage is neither counted
against a VM nor read as the VM ending. A VM is kept if it has a reading in at least 90%
of the slots in which the collector was up. Below that, too much of its work would have
to be guessed.

In [15]:
con.execute('''
    CREATE TABLE grid AS
    WITH life AS (SELECT vm, vcpu, unnest(range(s0, s1 + 1)) AS slot FROM keep WHERE NOT unmonitored)
    SELECT l.vm, l.vcpu, l.slot, g.slot IS NOT NULL AS up, u.cpu
    FROM life l LEFT JOIN gs g ON g.slot = l.slot
    LEFT JOIN u ON u.vm = l.vm AND u.slot = l.slot''')
before = scalar(con, "SELECT count(*) FROM keep")
con.execute(f'''
    DELETE FROM keep WHERE NOT unmonitored AND vm IN (
        SELECT vm FROM grid GROUP BY vm
        HAVING count(cpu) < {wc["min_usage_coverage"]} * count(*) FILTER (up))''')
log.step(f"usage coverage >= {wc['min_usage_coverage']:.0%}", before, scalar(con, "SELECT count(*) FROM keep"))

  usage coverage >= 90%                               767 ->        742 VMs  


### 2.5 Filling gaps inside a VM's life

Each slot of a measured VM's life ends up in one of four states:

| `fill` | When | Value | Why |
|---|---|---|---|
| `observed` | usage was reported | as reported | |
| `interpolated` | a gap of at most 3 slots (15 minutes) | linear between the readings either side | usage changes little over so short a gap |
| `outage_mean` | the collector was down for every VM | the VM's own mean usage | the VM was alive but unobserved. Skipping these slots would undercount its work $W_i$ |
| *(left out)* | a longer gap while the collector was up | not part of the work window $T^W_i$ | nothing nearby supports a value, and filling it would invent work |

Values are clipped at 100% per vCPU, the most a VM can use. The clip is written so that
a missing value stays missing: a clip that treated missing as "no limit reached" would
turn every long gap into full load.

In [16]:
con.execute(f'''
    CREATE TABLE usage AS
    WITH w AS (
        SELECT g.*,
            last_value(cpu IGNORE NULLS) OVER p AS c0,
            last_value(CASE WHEN cpu IS NOT NULL THEN slot END IGNORE NULLS) OVER p AS k0,
            first_value(cpu IGNORE NULLS) OVER n AS c1,
            first_value(CASE WHEN cpu IS NOT NULL THEN slot END IGNORE NULLS) OVER n AS k1
        FROM grid g WHERE vm IN (SELECT vm FROM keep)
        WINDOW p AS (PARTITION BY vm ORDER BY slot ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
               n AS (PARTITION BY vm ORDER BY slot ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING)),
    r AS (
        -- not least(): DuckDB's least() skips NULLs, which would turn every long gap
        -- into full load instead of leaving it out
        SELECT vm, slot, vcpu, up, cpu, CASE
                   WHEN cpu IS NOT NULL THEN cpu
                   WHEN k1 - k0 - 1 <= {wc["max_fill_slots"]} THEN c0 + (c1 - c0) * (slot - k0) / (k1 - k0)
               END AS raw
        FROM w),
    f AS (SELECT *, CASE WHEN raw > 100.0 * vcpu THEN 100.0 * vcpu ELSE raw END AS v FROM r)
    SELECT vm, slot, vcpu,
           coalesce(v, CASE WHEN NOT up THEN avg(v) OVER (PARTITION BY vm) END) AS cpu_pct,
           CASE WHEN cpu IS NOT NULL THEN 'observed' WHEN v IS NOT NULL THEN 'interpolated'
                WHEN NOT up THEN 'outage_mean' END AS fill
    FROM f''')
n_long = scalar(con, "SELECT count(*) FROM usage WHERE cpu_pct IS NULL")
con.execute("DELETE FROM usage WHERE cpu_pct IS NULL")
print(f"long gaps left out of the work window: {n_long:,} slots")
con.sql("SELECT fill, count(*) AS slots, count(DISTINCT vm) AS vms FROM usage GROUP BY 1 ORDER BY 2 DESC")

long gaps left out of the work window: 12,628 slots


┌──────────────┬─────────┬───────┐
│     fill     │  slots  │  vms  │
│   varchar    │  int64  │ int64 │
├──────────────┼─────────┼───────┤
│ observed     │ 2758428 │   502 │
│ outage_mean  │   29543 │   108 │
│ interpolated │     989 │   174 │
└──────────────┴─────────┴───────┘

### 2.6 Impute usage for VMs on never-reporting hosts

Controlled by `impute_unmonitored` in the settings cell, and **on** as shipped. VMs on
never-reporting hosts have a known arrival time, flavor and lifetime, but no usage.
Dropping them would be the cautious choice, because any usage given to them is borrowed,
not measured. But it removes a large share of the VMs, and the rest fill only about half
of the memory of ten servers, short of the normal and heavy days.

When the switch is on, each such VM takes the mean usage of a measured VM with the same
flavor, drawn at random (seeded) from the `impute_donor_pool` measured VMs nearest in
lifetime. Imputed workloads are flagged `usage_imputed = 1` in every output file, so
results can be reported with and without them.

When the switch is off, those VMs were already dropped in section 2.3, and this cell
does nothing.

In [17]:
con.execute("CREATE TABLE donor (vm VARCHAR, donor_vm VARCHAR)")
if impute:
    rng = random.Random(wc["impute_seed"])
    donors = rows(con, "SELECT vm, flavor_name, s1 - s0 + 1 AS life FROM keep WHERE NOT unmonitored")
    pairs = []
    for r in rows(con, "SELECT vm, flavor_name, s1 - s0 + 1 AS life FROM keep WHERE unmonitored ORDER BY vm"):
        pool = sorted((d for d in donors if d["flavor_name"] == r["flavor_name"]),
                      key=lambda d: (abs(math.log(d["life"] / r["life"])), d["vm"]))
        if pool:
            pairs.append((r["vm"], rng.choice(pool[:wc["impute_donor_pool"]])["vm"]))
    con.executemany("INSERT INTO donor VALUES (?, ?)", pairs)
    con.execute('''
        INSERT INTO usage
        SELECT k.vm, unnest(range(k.s0, k.s1 + 1)), k.vcpu, m.mean_cpu, 'imputed'
        FROM keep k JOIN donor d USING (vm)
        JOIN (SELECT vm, avg(cpu_pct) AS mean_cpu FROM usage GROUP BY vm) m ON m.vm = d.donor_vm''')
    before = scalar(con, "SELECT count(*) FROM keep")
    con.execute("DELETE FROM keep WHERE unmonitored AND vm NOT IN (SELECT vm FROM donor)")
    log.step("unmonitored VMs given a donor's usage", before, scalar(con, "SELECT count(*) FROM keep"),
             f"{len(pairs)} imputed; dropped if no same-flavor donor")
else:
    print("imputation off: VMs on unmonitored hosts were dropped")

  unmonitored VMs given a donor's usage               742 ->        741 VMs  239 imputed; dropped if no same-flavor donor


### 2.7 Workload parameters

| Parameter | Rule |
|---|---|
| $C^{req}_i$, $M^{req}_i$ | vCPUs and memory (MB) of the VM's flavor |
| $A_i$ | first slot the VM is seen. Time of day is kept in Campina Grande time (UTC-3) |
| $U^{CPU}_{it}$ | usage ÷ 100 × 2,400 MHz |
| $W_i$ | Eq. (14): $\sum_{t \in T^W_i} U^{CPU}_{it}\,\Delta t$ |
| $ET^{base}_i$ | $W_i / (C^{req}_i \cdot f_{ref})$, rounded up to whole periods |

UFCG reports usage as a percentage of one vCPU, while the model measures work in MHz.
One vCPU at 100% is taken as $f_{ref}$ = 2,400 MHz, the middle of the model's three
server frequencies, so that Eq. (14) keeps its units. $ET^{base}_i$ is the time a
workload needs on its own requested cores at $f_{ref}$ with no throttling.

The tables below describe the cleaned catalogue: workloads per flavor, and how baseline
execution time is distributed.

In [18]:
off = wc["utc_offset_hours"] * 3600
con.execute(f'''
    CREATE TABLE workloads AS
    WITH agg AS (
        SELECT vm, count(*) AS n_work_slots, count(*) FILTER (fill = 'outage_mean') AS n_outage_slots,
               sum(cpu_pct) / 100.0 * {slot} AS core_seconds,
               avg(cpu_pct) AS mean_cpu_pct, max(cpu_pct) AS peak_cpu_pct
        FROM usage GROUP BY vm)
    SELECT printf('w%04d', row_number() OVER (ORDER BY k.s0, k.vm)) AS workload_id,
           k.vm AS ufcg_server_id, k.flavor_name,
           k.vcpu AS C_req_vcpu, k.ram_mb AS M_req_MB,
           k.s0 * {slot} AS arrival_unix,
           CAST(((k.s0 * {slot} + {off}) % 86400) / {slot} AS INTEGER) AS arrival_slot_of_day,
           k.s1 - k.s0 + 1 AS lifetime_slots,
           a.n_work_slots, a.n_outage_slots,
           round(a.mean_cpu_pct / (100.0 * k.vcpu), 5) AS mean_util,
           round(a.peak_cpu_pct / (100.0 * k.vcpu), 5) AS peak_util,
           round(a.core_seconds, 2) AS core_seconds,
           round(a.core_seconds * {f_ref}, 1) AS W_MHz_s,
           -- time to deliver W_i on its own requested vCPUs at f_ref, unthrottled
           greatest(1, CAST(ceil(a.core_seconds / (k.vcpu * {slot})) AS INTEGER)) AS ET_base_periods,
           k.unmonitored::INTEGER AS usage_imputed, d.donor_vm AS usage_donor_ufcg_id
    FROM keep k JOIN agg a USING (vm) LEFT JOIN donor d USING (vm)''')
con.execute(f"COPY (SELECT * FROM workloads ORDER BY workload_id) TO '{CLEAN / 'workloads.csv'}' (HEADER)")
con.execute(f'''
    COPY (SELECT w.workload_id, u.slot - w.arrival_unix / {slot} AS k,
                 round(u.cpu_pct, 3) AS cpu_pct, round(u.cpu_pct / 100.0 * {f_ref}, 2) AS U_cpu_MHz,
                 u.fill <> 'observed' AS filled, u.fill
          FROM usage u JOIN workloads w ON w.ufcg_server_id = u.vm ORDER BY 1, 2)
    TO '{CLEAN / 'workload_usage.parquet'}' (FORMAT parquet)''')

log.entries = rlog.entries + log.entries
log.write(CLEAN / "cleaning_log_workloads.csv")
print(f"wrote {scalar(con, 'SELECT count(*) FROM workloads')} workloads")
con.sql('''
    SELECT CASE WHEN flavor_name LIKE 'reservation%' THEN 'reservations' ELSE flavor_name END AS flavor,
           min(C_req_vcpu) AS vcpus_min, max(C_req_vcpu) AS vcpus_max, count(*) AS workloads,
           sum(usage_imputed) AS imputed,
           round(median(mean_util), 4) AS median_util, median(ET_base_periods) AS median_ET_periods
    FROM workloads GROUP BY 1 ORDER BY workloads DESC''')

wrote 741 workloads


┌────────────────┬───────────┬───────────┬───────────┬─────────┬─────────────┬───────────────────┐
│     flavor     │ vcpus_min │ vcpus_max │ workloads │ imputed │ median_util │ median_ET_periods │
│    varchar     │   int64   │   int64   │   int64   │ int128  │   double    │      double       │
├────────────────┼───────────┼───────────┼───────────┼─────────┼─────────────┼───────────────────┤
│ general.medium │         2 │         2 │       318 │     119 │      0.0406 │              16.0 │
│ general.large  │         4 │         4 │       208 │      57 │      0.0527 │              10.0 │
│ general.small  │         1 │         1 │       168 │      46 │      0.0103 │              15.5 │
│ general.xlarge │         8 │         8 │        40 │      17 │      0.0534 │              18.0 │
│ reservations   │         1 │        16 │         7 │       0 │       0.017 │             220.0 │
└────────────────┴───────────┴───────────┴───────────┴─────────┴─────────────┴───────────────────┘

In [19]:
con.sql('''
    SELECT CASE WHEN ET_base_periods < 3 THEN '1: under 3 periods'
                WHEN ET_base_periods <= 12 THEN '2: 3-12 periods (up to 1 h)'
                WHEN ET_base_periods <= 144 THEN '3: 13-144 periods (up to 12 h)'
                WHEN ET_base_periods <= 288 THEN '4: 145-288 periods (up to 1 day)'
                ELSE '5: more than 1 day' END AS baseline_execution_time,
           count(*) AS workloads
    FROM workloads GROUP BY 1 ORDER BY 1''')

┌──────────────────────────────────┬───────────┐
│     baseline_execution_time      │ workloads │
│             varchar              │   int64   │
├──────────────────────────────────┼───────────┤
│ 1: under 3 periods               │       199 │
│ 2: 3-12 periods (up to 1 h)      │       160 │
│ 3: 13-144 periods (up to 12 h)   │       179 │
│ 4: 145-288 periods (up to 1 day) │        49 │
│ 5: more than 1 day               │       154 │
└──────────────────────────────────┴───────────┘

## 3. Model inputs

Cleaning ends with section 2. This part selects from the two cleaned tables and writes
one folder of model inputs per scenario defined in the settings cell. Nothing here
changes the cleaned data.

### The scenarios

The settings cell holds one set of **reference settings**, `servers10`: the data center as
the thesis describes it, ten servers of two models: three Dell R840 at 2.1 GHz, four R840 at
2.4 GHz and three Dell C6320 at 2.6 GHz, in two cooling zones, with cold plates on the
C6320s, over a 24-hour horizon from 06:00, on a normal day. Every other scenario is
these settings with some values changed, and its `params.json` records the changes under
`variant_of`.

| Scenario | Changed from the reference settings | Question it answers |
|---|---|---|
| `servers10` | nothing: the reference settings on a normal day, workloads drawn until peak demand reaches 60% of the fleet's cores or memory | The data center of the thesis text, in the Chapter 6 runs. |
| `servers6` | 6 servers: two R840 at 2.1 GHz, two R840 at 2.4 GHz, two C6320 at 2.6 GHz | The smaller fleet of the Chapter 6 runs. |
| `servers10_light`, `servers10_heavy`, `servers6_light`, `servers6_heavy` | the same fleet on a light or heavy day: peak demand 30% or 85% | How do the savings change with how busy the day is? |
| `small` | 3 servers (one of each: R840 2.1 GHz, R840 2.4 GHz, C6320 2.6 GHz), 8 h from 12:00, shorter workloads and slack | None about the data center. It is small enough for quick solver checks and debugging. |
| `small_running` | `small` with the workloads arriving in periods 3-5 already running at the start, and server 3 down for one hour | None about the data center. It exercises Eq. (11) and (29), which no other scenario makes binding. It is made from `small`'s files, not from the cleaned tables. |

An R840 has 64 cores and 64 GB of memory, a C6320 80 cores and 193 GB. Almost every
workload asks for 2 GB per vCPU, so across the fleet memory, not cores, fills first: the
load levels are reached on memory.

Other fleets (two 2.4 GHz R840s, for example) are one entry each in the `fleets` dict of
the settings cell, as a list of server rows. Some experiments need no scenario of their own, because they change
how a scenario is solved rather than what it contains: a flat carbon intensity, regrouped
cooling zones, cold plates on more servers, and a different minimum thermal headroom. They
are settings of the solve, so this notebook does not build them.

### How a scenario is built

| File | Content |
|---|---|
| `servers.csv` | one server per row of the fleet list, its host drawn by seed from the eligible hosts of that row's model, without regard to power; frequency, zone and cold plate as the row gives them |
| `workloads.csv` | workloads drawn in seeded random order until peak requested vCPUs reach the target share of fleet cores, or peak requested memory that share of fleet memory, whichever comes first; if the catalogue runs out first, it is drawn again in the same order (IDs ending `-r1`, `-r2`, ...) |
| `carbon.csv` | carbon intensity per period |
| `baseline_schedule.csv` | a reference schedule that starts each workload at arrival on the least-loaded server with room |
| `params.json` | settings, constants and summary figures |

Cleaned workloads arrive across a whole year, too sparsely to load a fleet, so arrivals
are pooled into one day with each workload's time of day kept.

The datasets do not supply deadlines, the time-sensitive flag, zones, cold plates, the
frequency of each server (taken from SPEC CPU results for the server model's processor) or
an hourly carbon profile. These are **assumed** by the rules
in the settings cell, and each `params.json` lists them under `assumed_not_from_data`.

The first cell below holds the construction rules. The second cell applies them to each
scenario of the settings cell, then makes `small_running` from `small`.

In [17]:
# Scenario construction rules: servers, workloads, carbon profile, baseline schedule.
import csv
import hashlib
import json
import math
import random
import shutil
from pathlib import Path


def read_csv_rows(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def write_csv(path, records, fields=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = fields or list(records[0].keys())
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(records)


def pick_servers(sc, clean_dir):
    pool = [s for s in read_csv_rows(Path(clean_dir) / "servers.csv") if s["eligible"] == "1"]
    for s in pool:
        for k, v in s.items():
            if k != "server":
                s[k] = float(v)
    pool.sort(key=lambda s: s["P_max_W"] - s["P_idle_W"])
    n = sc["n_servers"]
    if n > len(pool):
        raise ValueError(f"n_servers = {n}, but only {len(pool)} servers in the cleaned table are eligible")
    chosen = [pool[int((i + 0.5) * len(pool) / n)] for i in range(n)]   # bin midpoints, no extremes

    freqs = [f for f, k in zip(sc["freq_mhz"], sc["freq_split"]) for _ in range(k)]
    top_f = max(sc["freq_mhz"])
    out = []
    for rank, (s, f) in enumerate(zip(sorted(chosen, key=lambda s: s["P_max_W"]), freqs)):
        out.append({
            "s": rank + 1, "server": s["server"], "zone": rank % sc["n_zones"] + 1,
            "f_s_MHz": f, "N_cores": int(s["N_cores"]), "C_s_cores": int(s["C_s_cores"]),
            "M_s_MB": round(s["M_s_MB"]), "PC_s_MHz": s["N_cores"] * f,
            "P_idle_W": round(s["P_idle_W"], 2), "P_max_W": round(s["P_max_W"], 2),
            "P_fit_full_load_W": round(s["P_fit_full_load_W"], 2),
            "T0_C": round(s["T0_C"], 2), "CT_s_cold_plate": int(f == top_f),
            "n_cold_plates": int(f == top_f),   # editable: plates on the server (0 = none)
            "th_a_C": round(s["th_a_C"], 3), "th_R_C_per_W": round(s["th_R_C_per_W"], 5),
            "th_alpha_5min": round(s["th_alpha_5min"], 5),
            # dynamic watts drawn per MHz of work delivered: lower is more efficient
            "W_per_MHz": round((s["P_max_W"] - s["P_idle_W"]) / (s["N_cores"] * f), 6),
        })
    return out


def is_time_sensitive(vm_id, seed, share):
    # Seeded hash of the VM id, uniform on [0, 1): depends only on this workload, so the
    # flag does not change when other workloads enter or leave the catalogue. Keyed on
    # the UFCG id because workload_id is a sort position in the catalogue and shifts with it.
    h = hashlib.sha256(f"{seed}:{vm_id}".encode()).digest()
    return int(int.from_bytes(h[:8], "big") / 2 ** 64 < share)


def pick_workloads(sc, fleet_cores, fleet_mem_mb, clean_dir):
    # Draw in seeded random order until peak requested vCPUs reach target_peak_load of the
    # fleet's cores, or peak requested memory that share of its memory, whichever comes first.
    # With top_up_by_repeats, a catalogue that runs out first is drawn again in the same
    # order: each copy keeps the workload's arrival time, work and flags, and its ID gets
    # "-r1", "-r2", ... (column `repeat`).
    T, rng = sc["periods"], random.Random(sc["seed"])
    per_day = 86400 // sc["dt_seconds"]
    start = sc["start_hour_local"] * 3600 // sc["dt_seconds"]
    cat = [w for w in read_csv_rows(Path(clean_dir) / "workloads.csv")
           if sc["min_et_periods"] <= int(w["ET_base_periods"]) <= sc["max_et_periods"]]
    rng.shuffle(cat)

    demand, mem, picked = [0] * (T + 2), [0] * (T + 2), []

    def full():
        return (max(demand) >= sc["target_peak_load"] * fleet_cores
                or max(mem) >= sc["target_peak_load"] * fleet_mem_mb)

    repeat = 0
    while True:
        n_before = len(picked)
        for w in cat:
            et, vcpu, mb = int(w["ET_base_periods"]), int(w["C_req_vcpu"]), int(w["M_req_MB"])
            a = (int(w["arrival_slot_of_day"]) - start) % per_day + 1
            rt = is_time_sensitive(w["ufcg_server_id"], sc["seed"], sc["rt_share"])
            slack = max(1, math.ceil(sc["rt_slack_frac"] * et)) if rt else \
                min(sc["dt_slack_max"], max(sc["dt_slack_min"], math.ceil(sc["dt_slack_mult"] * et)))
            d = min(a + et + slack, T + 1)          # D_i: first period by which it must be done
            if d - a - et < 1:
                continue                            # cannot finish inside the horizon
            for t in range(a, a + et):
                demand[t] += vcpu
                mem[t] += mb
            picked.append({
                "workload_id": w["workload_id"] + (f"-r{repeat}" if repeat else ""),
                "flavor": w["flavor_name"],
                "A_i": a, "D_i": d, "AD_i": d - a - et, "ET_base_i": et, "RT_i": rt, "B_i": 0,
                "C_req_i": vcpu, "M_req_i_MB": mb, "W_i_MHz_s": float(w["W_MHz_s"]),
                "usage_imputed": int(w["usage_imputed"]), "repeat": repeat,
            })
            if full():
                break
        if full() or len(picked) == n_before or not sc.get("top_up_by_repeats", False):
            break
        repeat += 1
    if not picked:
        raise ValueError("no workload fits: none can finish inside the horizon with these settings")
    picked.sort(key=lambda w: (w["A_i"], w["workload_id"]))
    for i, w in enumerate(picked, 1):
        w["i"] = i
    return picked, max(demand), max(mem), len(cat)


def carbon_profile(sc):
    # Annual mean with a synthetic daily shape: a midday dip and an evening peak.
    T, dt = sc["periods"], sc["dt_seconds"]
    hours = [(sc["start_hour_local"] + (t + 0.5) * dt / 3600) % 24 for t in range(T)]
    bump = lambda h, mu, sd: math.exp(-0.5 * (min(abs(h - mu), 24 - abs(h - mu)) / sd) ** 2)
    day = [bump(h / 12, 19, 2.0) - bump(h / 12, 12, 3.0) for h in range(288)]
    mean, span = sum(day) / 288, max(abs(x - sum(day) / 288) for x in day)
    shape = lambda h: (bump(h, 19, 2.0) - bump(h, 12, 3.0) - mean) / span
    return [{"t": t + 1, "hour_local": round(h, 3),
             "CI_t_g_per_kWh": round(sc["ci_mean_g_per_kwh"] * (1 + sc["ci_amplitude"] * shape(h)), 2)}
            for t, h in enumerate(hours)]


def baseline_schedule(servers, workloads, sc, unavailable=frozenset()):
    # Thermally unaware reference: each workload starts at arrival on the least-loaded
    # server that has room for its cores and memory, Eq. (12)-(13). `unavailable` holds
    # (server, period) pairs with AV = 0, Eq. (11).
    T, dt = sc["periods"], sc["dt_seconds"]
    used = {s["s"]: [0] * (T + 2) for s in servers}
    mem = {s["s"]: [0] * (T + 2) for s in servers}
    sched = []
    for w in workloads:
        placed = None
        for start in range(w["A_i"], T + 1):
            for s in sorted(servers, key=lambda s: (used[s["s"]][start], s["s"])):
                et = math.ceil(w["W_i_MHz_s"] / (w["C_req_i"] * s["f_s_MHz"] * dt))
                span = range(start, start + max(1, et))
                if span[-1] <= T and all(used[s["s"]][t] + w["C_req_i"] <= s["C_s_cores"]
                                         and mem[s["s"]][t] + w["M_req_i_MB"] <= s["M_s_MB"]
                                         and (s["s"], t) not in unavailable for t in span):
                    placed = (s, span)
                    break
            if placed:
                break
        if not placed:
            sched.append({"i": w["i"], "s": "", "start": "", "end": "", "late": 1})
            continue
        s, span = placed
        for t in span:
            used[s["s"]][t] += w["C_req_i"]
            mem[s["s"]][t] += w["M_req_i_MB"]
        w["X0_s"] = s["s"]
        sched.append({"i": w["i"], "s": s["s"], "start": span[0], "end": span[-1] + 1,
                      "late": int(span[-1] + 1 > w["D_i"])})
    return sched, used


def check_settings(sc):
    """The scenario settings must describe a fleet the builder can make."""
    if len(sc["freq_mhz"]) != len(sc["freq_split"]):
        raise ValueError(f"freq_mhz has {len(sc['freq_mhz'])} entries but freq_split has {len(sc['freq_split'])}")
    if sum(sc["freq_split"]) != sc["n_servers"]:
        raise ValueError(f"freq_split {sc['freq_split']} adds up to {sum(sc['freq_split'])}, "
                         f"not n_servers = {sc['n_servers']}")
    if not 1 <= sc["n_zones"] <= sc["n_servers"]:
        raise ValueError(f"n_zones = {sc['n_zones']} must be between 1 and n_servers = {sc['n_servers']}")
    if sc["periods"] < 1 or sc["min_et_periods"] > sc["max_et_periods"]:
        raise ValueError("periods must be at least 1 and min_et_periods at most max_et_periods")


def build_scenario(name, sc, const, cold_plate, *, clean_dir, scenarios_dir, f_ref_mhz,
                   slot_seconds=300, extra=None):
    """Write scenarios_dir/name/ from scenario settings ``sc``, the constants and the
    cold-plate table. ``extra`` is added to params.json. Returns (servers, diagnostics)."""
    assert sc["dt_seconds"] == slot_seconds, "period length must match the UFCG slot"
    check_settings(sc)
    out = Path(scenarios_dir) / name
    T, dt = sc["periods"], sc["dt_seconds"]

    servers = pick_servers(sc, clean_dir)
    fleet = sum(s["C_s_cores"] for s in servers)
    fleet_mem = sum(s["M_s_MB"] for s in servers)
    workloads, peak, peak_mem, n_cat = pick_workloads(sc, fleet, fleet_mem, clean_dir)
    carbon = carbon_profile(sc)
    sched, used = baseline_schedule(servers, workloads, sc)

    # Baseline IT energy and carbon by Eq. (16), (20), (3), (6); cooling is not included.
    ci = [c["CI_t_g_per_kWh"] for c in carbon]
    kwh = [sum(s["P_idle_W"] + (s["P_max_W"] - s["P_idle_W"]) * used[s["s"]][t] / s["C_s_cores"]
               for s in servers) * dt / 3.6e6 for t in range(1, T + 1)]
    cap_T = const["T_max_server_C"] - const["H_min_C"]

    def u_ceiling(s, inlet):
        # Highest steady utilization that keeps the server at or under T_max - H_min.
        p = (cap_T - inlet - s["th_a_C"]) / s["th_R_C_per_W"]
        return round(max(0.0, min(1.0, (p - s["P_idle_W"]) / (s["P_max_W"] - s["P_idle_W"]))), 3)

    for s in servers:
        s["U_ceiling_inlet_min"] = u_ceiling(s, const["T_crac_min_C"])
        s["U_ceiling_inlet_max"] = u_ceiling(s, const["T_crac_max_C"])

    write_csv(out / "servers.csv", servers)
    write_csv(out / "workloads.csv", workloads, ["i", "workload_id", "flavor", "A_i", "D_i", "AD_i", "ET_base_i",
                                                 "RT_i", "B_i", "X0_s", "C_req_i", "M_req_i_MB", "W_i_MHz_s",
                                                 "usage_imputed", "repeat"])
    write_csv(out / "carbon.csv", carbon)
    write_csv(out / "baseline_schedule.csv", sched)

    top_f = max(sc["freq_mhz"])
    top_cores = sum(s["C_s_cores"] for s in servers if s["f_s_MHz"] == top_f)   # 0: no server of that tier
    diag = {
        "I_workloads": len(workloads), "S_servers": len(servers), "Z_zones": sc["n_zones"], "T_periods": T,
        # Y_ist can only be 1 between arrival and deadline, so this is the real count
        "Y_ist_binaries_inside_windows": sum(w["D_i"] - w["A_i"] for w in workloads) * len(servers),
        "catalogue_size_within_ET_limits": n_cat,
        "fleet_cores": fleet,
        "peak_requested_vcpus": peak,
        "peak_load_share_of_fleet": round(peak / fleet, 3),
        "peak_load_share_of_fastest_tier": round(peak / top_cores, 3) if top_cores else None,
        "fleet_memory_MB": fleet_mem,
        "peak_requested_memory_MB": peak_mem,
        "peak_memory_share_of_fleet": round(peak_mem / fleet_mem, 3),
        "time_sensitive_workloads": sum(w["RT_i"] for w in workloads),
        "workloads_with_imputed_usage": sum(w["usage_imputed"] for w in workloads),
        "repeated_workloads": sum(w["repeat"] > 0 for w in workloads),
        "median_slack_periods_delay_tolerant": sorted(w["AD_i"] for w in workloads if not w["RT_i"])[
            sum(1 for w in workloads if not w["RT_i"]) // 2] if any(not w["RT_i"] for w in workloads) else None,
        "W_per_MHz_spread_best_to_worst": round(max(s["W_per_MHz"] for s in servers) / min(s["W_per_MHz"] for s in servers), 2),
        "servers_thermally_capped_at_coldest_inlet": sum(s["U_ceiling_inlet_min"] < 1 for s in servers),
        "servers_thermally_capped_at_warmest_inlet": sum(s["U_ceiling_inlet_max"] < 1 for s in servers),
        "CI_min_max_g_per_kWh": [min(ci), max(ci)],
        "baseline_IT_energy_kWh": round(sum(kwh), 2),
        # idle draw is paid whatever the schedule is: the model has no server on/off variable
        "baseline_idle_share_of_IT_energy": round(sum(s["P_idle_W"] for s in servers) * T * dt / 3.6e6 / sum(kwh), 3),
        "baseline_IT_carbon_kg": round(sum(k * c for k, c in zip(kwh, ci)) / 1000, 2),
        "baseline_unplaced_or_late": sum(x["late"] for x in sched),
    }
    params = {
        "scenario": name, "settings": sc, "constants_as_printed": const,
        "cold_plate": cold_plate,
        "time": {"dt_seconds": dt, "T": T, "period_1_local_hour": sc["start_hour_local"],
                 "convention": "A_i = first period it may run; D_i = period by which it must be finished "
                               "(CT_i = ST_i + ET_i <= D_i); AD_i = D_i - A_i - ET_base_i, Eq. (22) as an equality"},
        "work_rule": "ET_is = W_i / (C_req_i * f_s * DPF * dt): a workload is served by its own requested "
                     "cores, not the whole server. ET_base_i uses f_ref.",
        "f_ref_MHz": f_ref_mhz,
        "thermal_fit": "steady state T_cpu = T_inlet + th_a_C + th_R_C_per_W * P_IT; "
                       "per period T[t] = alpha*T[t-1] + (1-alpha)*steady, alpha = th_alpha_5min. "
                       "Fitted from UniMelb; not part of the thesis formulation.",
        "assumed_not_from_data": ["RT_i", "D_i", "zone", "CT_s_cold_plate", "n_cold_plates",
                                  "f_s_MHz assignment",
                                  "daily shape of CI_t", "AV_st = 1 for all s, t", "B_i = 0"],
        "diagnostics": diag,
        **(extra or {}),
        "built_with": {"thermasched": VERSION},
    }
    with open(out / "params.json", "w") as f:
        json.dump(params, f, indent=2)
    return servers, diag


def build_running_variant(name, start_from, running_arrivals, outage, *, scenarios_dir):
    """Write scenarios_dir/name/ as scenario ``start_from`` with some workloads already running
    (B_i = 1 for those arriving in periods ``running_arrivals`` = [first, last], Eq. 29) and
    one server down (``outage`` = {"server": s, "periods": [first, last]}, AV = 0, Eq. 11).
    The baseline schedule is rebuilt by the same least-loaded rule, skipping the downtime."""
    src, out = Path(scenarios_dir) / start_from, Path(scenarios_dir) / name
    with open(src / "params.json") as f:
        params = json.load(f)
    T, dt = params["time"]["T"], params["time"]["dt_seconds"]
    server_rows = read_csv_rows(src / "servers.csv")
    rows = read_csv_rows(src / "workloads.csv")
    servers = [{"s": int(r["s"]), "f_s_MHz": float(r["f_s_MHz"]), "C_s_cores": int(r["C_s_cores"]),
                "M_s_MB": int(r["M_s_MB"])} for r in server_rows]
    workloads = [{"i": int(r["i"]), "A_i": int(r["A_i"]), "D_i": int(r["D_i"]),
                  "C_req_i": int(r["C_req_i"]), "M_req_i_MB": int(r["M_req_i_MB"]),
                  "W_i_MHz_s": float(r["W_i_MHz_s"])} for r in rows]

    first, last = running_arrivals
    running = [r for r in rows if first <= int(r["A_i"]) <= last]
    for r in running:
        r["B_i"] = 1
    down_s, (t0, t1) = outage["server"], outage["periods"]
    if not 1 <= down_s <= len(servers) or not 1 <= t0 <= t1 <= T:
        raise ValueError(f"outage {outage} is outside the {len(servers)} servers or {T} periods")
    down = {(down_s, t) for t in range(t0, t1 + 1)}
    sched, _ = baseline_schedule(servers, workloads, {"periods": T, "dt_seconds": dt}, down)
    for r, w in zip(rows, workloads):
        r["X0_s"] = w.get("X0_s", "")

    out.mkdir(parents=True, exist_ok=True)
    for fname in ("servers.csv", "carbon.csv"):
        shutil.copyfile(src / fname, out / fname)
    write_csv(out / "workloads.csv", rows)
    write_csv(out / "baseline_schedule.csv", sched)
    write_csv(out / "availability.csv", [{"s": s, "t": t, "AV": 0} for s, t in sorted(down)])

    params.pop("variant_of", None)
    params.pop("built_with", None)
    params["scenario"] = name
    params["assumed_not_from_data"] = [
        {"AV_st = 1 for all s, t": "AV_st (availability.csv)", "B_i = 0": "B_i (first arrivals)"}.get(a, a)
        for a in params["assumed_not_from_data"]]
    params["derived_from"] = {
        "scenario": start_from,
        "changes": [
            f"B_i = 1 for the {len(running)} workloads arriving in periods {first}-{last} (already "
            f"running at the start, so they keep their server, Eq. 29)",
            f"availability.csv: server {down_s} has AV = 0 in periods {t0}-{t1} "
            f"({(t1 - t0 + 1) * dt / 3600:g} h of maintenance, Eq. 11)",
            "baseline_schedule.csv and X0_s rebuilt with the least-loaded rule, which skips "
            "unavailable periods",
        ],
        "purpose": "exercise Eq. (11) and (29), which are never binding in the other scenarios",
    }
    params["built_with"] = {"thermasched": VERSION}
    with open(out / "params.json", "w") as f:
        json.dump(params, f, indent=2)
    return sched


# A scenario's inputs, in the order they enter its fingerprint.
FINGERPRINT_FILES = ("params.json", "servers.csv", "workloads.csv", "carbon.csv",
                     "baseline_schedule.csv", "availability.csv")


def scenario_fingerprint(folder):
    """Short hash of a scenario folder's inputs: equal fingerprints mean the same scenario.
    params.json is compared by content without ``built_with``, and line endings are
    ignored, so a rebuild with a newer release or on another system still matches.
    None if the folder has no params.json."""
    folder = Path(folder)
    if not (folder / "params.json").is_file():
        return None
    h = hashlib.sha256()
    for fname in FINGERPRINT_FILES:
        path = folder / fname
        if not path.is_file():
            continue
        if fname == "params.json":
            params = json.loads(path.read_text())
            params.pop("built_with", None)
            data = json.dumps(params, sort_keys=True).encode()
        else:
            data = path.read_bytes().replace(b"\r\n", b"\n")
        h.update(fname.encode() + b"\0" + data + b"\0")
    return h.hexdigest()[:12]


In [18]:
# Each scenario is servers10's settings with some values changed; its params.json records which
main = servers10
results = {}
for name, sc in cfg["scenario"].items():
    const_changes = cfg["scenario_constants"].get(name, {})
    changed = {"settings": {k: v for k, v in sc.items() if v != main[k]}, "constants": const_changes}
    extra = None if name == "servers10" else {
        "variant_of": {"scenario": "servers10", "changed": {k: v for k, v in changed.items() if v}}}
    results[name] = build_scenario(name, sc, cfg["constants"] | const_changes, cfg["cold_plate"],
                                   clean_dir=CLEAN, scenarios_dir=ROOT / cfg["paths"]["scenarios"],
                                   f_ref_mhz=cfg["workloads"]["f_ref_mhz"],
                                   slot_seconds=cfg["workloads"]["slot_seconds"], extra=extra)
    print(f"{name:<22} " + ("reference settings" if extra is None else f"servers10 with {extra['variant_of']['changed']}"))
for name, v in cfg["running_variants"].items():
    build_running_variant(name, v["from"], v["running_arrivals"], v["outage"],
                          scenarios_dir=ROOT / cfg["paths"]["scenarios"])
    print(f"{name:<22} {v['from']} with workloads already running and server {v['outage']['server']} down")


servers10_light        servers10 with {'settings': {'target_peak_load': 0.3}}
servers10_light_swing  servers10 with {'settings': {'target_peak_load': 0.3, 'ci_amplitude': 0.5}}
servers10              reference settings
servers10_swing        servers10 with {'settings': {'ci_amplitude': 0.5}}
servers10_heavy        servers10 with {'settings': {'target_peak_load': 0.85}}
servers10_heavy_swing  servers10 with {'settings': {'target_peak_load': 0.85, 'ci_amplitude': 0.5}}
servers6_light         servers10 with {'settings': {'n_servers': 6, 'freq_split': [2, 2, 2], 'target_peak_load': 0.3}}
servers6_light_swing   servers10 with {'settings': {'n_servers': 6, 'freq_split': [2, 2, 2], 'target_peak_load': 0.3, 'ci_amplitude': 0.5}}
servers6               servers10 with {'settings': {'n_servers': 6, 'freq_split': [2, 2, 2]}}
servers6_swing         servers10 with {'settings': {'n_servers': 6, 'freq_split': [2, 2, 2], 'ci_amplitude': 0.5}}
servers6_heavy         servers10 with {'settings': {'n_serve

In [19]:
# Scenario diagnostics side by side
names = list(results)
print(f"{'':<44}" + "".join(f"{n:>23}" for n in names))
for k in results[names[0]][1]:
    print(f"{k:<44}" + "".join(f"{str(results[n][1][k]):>23}" for n in names))

                                                    servers10_light  servers10_light_swing              servers10        servers10_swing        servers10_heavy  servers10_heavy_swing         servers6_light   servers6_light_swing               servers6         servers6_swing         servers6_heavy   servers6_heavy_swing                  small
I_workloads                                                      91                     91                    257                    257                    370                    370                     45                     45                    132                    132                    210                    210                     79
S_servers                                                        10                     10                     10                     10                     10                     10                      6                      6                      6                      6                      6               

In [20]:
# The servers6 fleet (the same servers on every load level)
print("s  server      zone  f_MHz  CP  P_idle  P_max  W/MHz     U-ceiling(inlet 18 / 27 C)")
for s in results["servers6"][0]:
    print(f"{s['s']:<2} {s['server']:<11} {s['zone']:<5} {s['f_s_MHz']:<6.0f} {s['CT_s_cold_plate']:<3} "
          f"{s['P_idle_W']:<7.1f} {s['P_max_W']:<6.1f} {s['W_per_MHz']:<9.5f} "
          f"{s['U_ceiling_inlet_min']} / {s['U_ceiling_inlet_max']}")

s  server      zone  f_MHz  CP  P_idle  P_max  W/MHz     U-ceiling(inlet 18 / 27 C)
1  qh2-rcc188  1     2100   0   144.2   312.0  0.00125   1.0 / 0.737
2  qh2-rcc158  2     2100   0   158.7   352.3  0.00144   1.0 / 0.764
3  qh2-rcc159  1     2400   0   158.7   360.0  0.00131   1.0 / 0.682
4  qh2-rcc181  2     2400   0   152.9   361.9  0.00136   0.949 / 0.631
5  qh2-rcc133  1     3000   1   152.8   368.0  0.00112   1.0 / 0.801
6  qh2-rcc138  2     3000   1   126.6   368.0  0.00126   0.927 / 0.614


## 4. Checks

Every assertion must pass. They confirm that:

- $W_i$ equals Eq. (14) summed over the stored usage series.
- Every stored slot has a known fill kind, and imputed slots appear only when
  imputation is switched on.
- In each scenario, arrival + baseline time + slack equals the deadline and stays inside
  the horizon.
- Every workload has a baseline server, and no ID repeats.
- Every time-sensitive flag follows the per-workload rule.
- In `small_running`, no workload runs on the server while it is down.

The second cell compares each scenario with the one the release was built with, by a
fingerprint of its files. **Same** means the scenario is identical to the reference, byte for
byte. **Different** is expected when a setting was changed; with the settings as shipped, it
means the raw files are not the same copy as the reference (see the size note in the
second code cell). A different scenario is still valid, but its results cannot be compared
with results solved on the reference scenarios.


In [21]:
chk = connect()
# Eq. (14): W_i = sum over T^W_i of U_it * dt. The stored U is rounded to 0.01 MHz, so each
# slot can be off by up to 0.005 MHz * dt, and W_i itself is rounded to 0.1. A flat imputed
# profile rounds the same way in every slot, so the bound is per slot, not relative.
bad = chk.sql(f'''
    SELECT count(*) FROM (
        SELECT w.workload_id, count(*) AS n, sum(u.U_cpu_MHz) * {slot} AS w14, any_value(w.W_MHz_s) AS w
        FROM '{CLEAN / "workload_usage.parquet"}' u JOIN '{CLEAN / "workloads.csv"}' w USING (workload_id)
        GROUP BY 1)
    WHERE abs(w14 - w) > 0.005 * {slot} * n + 0.05''').fetchone()[0]
assert bad == 0, f"{bad} workloads where W_i does not match Eq. (14)"

fills = dict(chk.sql(f"SELECT fill, count(*) FROM '{CLEAN / 'workload_usage.parquet'}' GROUP BY 1").fetchall())
assert set(fills) <= {"observed", "interpolated", "outage_mean", "imputed"}, fills
assert (impute or "imputed" not in fills), "imputed rows present with imputation off"

for name in results:
    d = ROOT / cfg["paths"]["scenarios"] / name
    T = cfg["scenario"][name]["periods"]
    ws = read_csv_rows(d / "workloads.csv")
    assert len({w["workload_id"] for w in ws}) == len(ws), f"{name}: repeated workload ID"
    for w in ws:
        a, dd, ad, et = (int(w[k]) for k in ("A_i", "D_i", "AD_i", "ET_base_i"))
        assert a + et + ad == dd <= T + 1, f"{name}: timing broken for {w['workload_id']}"
        assert w["X0_s"], f"{name}: {w['workload_id']} has no baseline server"
    vm_of = {r["workload_id"]: r["ufcg_server_id"] for r in read_csv_rows(CLEAN / "workloads.csv")}
    sc_ = cfg["scenario"][name]
    assert all(int(w["RT_i"]) == is_time_sensitive(vm_of[w["workload_id"].split("-r")[0]], sc_["seed"],
                                                    sc_["rt_share"])
               for w in ws), f"{name}: RT_i does not follow the per-workload hash"
for name, v in cfg["running_variants"].items():
    down_s, (t0, t1) = v["outage"]["server"], v["outage"]["periods"]
    sched = read_csv_rows(ROOT / cfg["paths"]["scenarios"] / name / "baseline_schedule.csv")
    assert not any(r["s"] and int(r["s"]) == down_s and int(r["start"]) <= t1 and int(r["end"]) > t0
                   for r in sched), f"{name}: a workload runs on server {down_s} while it is down"
print("all checks passed;", fills)

all checks passed; {'interpolated': 989, 'imputed': 1481247, 'outage_mean': 29543, 'observed': 2758428}


In [22]:
# Each scenario against the reference of this release
written = list(cfg["scenario"]) + list(cfg["running_variants"])
print(f"{'scenario':<22}{'fingerprint':>14}{'reference':>14}  release {VERSION}")
differ = []
for name in written:
    fp, ref_fp = scenario_fingerprint(ROOT / cfg["paths"]["scenarios"] / name), REFERENCE_FINGERPRINTS.get(name)
    verdict = "same" if fp == ref_fp else "no reference" if ref_fp is None else "DIFFERENT"
    differ += [name] if verdict == "DIFFERENT" else []
    print(f"{name:<22}{fp:>14}{ref_fp or '-':>14}  {verdict}")
if differ:
    print(f"\n{len(differ)} scenario(s) differ from the reference. Expected if a setting was changed; "
          "otherwise the raw files are another copy. See the text above.")


scenario                 fingerprint     reference  release 1.3.0
servers10_light         25f23c3dfb79  25f23c3dfb79  same
servers10_light_swing   dc1018a911c2  dc1018a911c2  same
servers10               3edf034fdf46  3edf034fdf46  same
servers10_swing         81dbb921dc62  81dbb921dc62  same
servers10_heavy         627e163ebe02  627e163ebe02  same
servers10_heavy_swing   bfcae8a67536  bfcae8a67536  same
servers6_light          cb0c78e1602e  cb0c78e1602e  same
servers6_light_swing    9c7b078d7cd7  9c7b078d7cd7  same
servers6                94352bda5aa7  94352bda5aa7  same
servers6_swing          fdc4e53b0393  fdc4e53b0393  same
servers6_heavy          9f9a249a2d1e  9f9a249a2d1e  same
servers6_heavy_swing    bde23dcd3ffb  bde23dcd3ffb  same
small                   ef6702bb949b  ef6702bb949b  same
small_running           42b739688470  42b739688470  same


## 5. Output reference

What the finished data looks like, and what each column is for. The cell below prints
the size of every output file from the run just completed. The tables after it describe
the columns.

Columns marked **model** are inputs the model reads, and the symbol is given. The others
are kept as evidence: they show why a server or workload was kept, or let a value be
traced back to the raw data.

In [20]:
ref = connect()
outputs = [CLEAN / f for f in ("servers.csv", "workloads.csv", "workload_usage.parquet",
                               "cleaning_log_servers.csv", "cleaning_log_workloads.csv")]
for name in list(cfg["scenario"]) + list(cfg["running_variants"]):
    d = ROOT / cfg["paths"]["scenarios"] / name
    outputs += [d / f for f in ("servers.csv", "workloads.csv", "carbon.csv", "baseline_schedule.csv",
                                "availability.csv") if (d / f).is_file()]
print(f"{'file':<56}{'rows':>12}{'columns':>9}")
for p in outputs:
    n_rows = scalar(ref, f"SELECT count(*) FROM '{p}'")
    n_cols = len(ref.sql(f"SELECT * FROM '{p}'").columns)
    print(f"{str(p.relative_to(ROOT)):<56}{n_rows:>12,}{n_cols:>9}")
ref.close()

file                                                            rows  columns
data/clean/servers.csv                                           145       23
data/clean/workloads.csv                                         741       17
data/clean/workload_usage.parquet                          4,270,207        6
data/clean/cleaning_log_servers.csv                                9        7
data/clean/cleaning_log_workloads.csv                             14        7
scenarios/servers10_light/servers.csv                             10       20
scenarios/servers10_light/workloads.csv                           91       15
scenarios/servers10_light/carbon.csv                             288        3
scenarios/servers10_light/baseline_schedule.csv                   91        5
scenarios/servers10_light_swing/servers.csv                       10       20
scenarios/servers10_light_swing/workloads.csv                     91       15
scenarios/servers10_light_swing/carbon.csv                      

scenarios/servers6_swing/carbon.csv                              288        3
scenarios/servers6_swing/baseline_schedule.csv                   132        5
scenarios/servers6_heavy/servers.csv                               6       20
scenarios/servers6_heavy/workloads.csv                           210       15
scenarios/servers6_heavy/carbon.csv                              288        3
scenarios/servers6_heavy/baseline_schedule.csv                   210        5
scenarios/servers6_heavy_swing/servers.csv                         6       20
scenarios/servers6_heavy_swing/workloads.csv                     210       15


scenarios/servers6_heavy_swing/carbon.csv                        288        3
scenarios/servers6_heavy_swing/baseline_schedule.csv             210        5
scenarios/small/servers.csv                                        3       20
scenarios/small/workloads.csv                                     79       15
scenarios/small/carbon.csv                                        96        3
scenarios/small/baseline_schedule.csv                             79        5
scenarios/small_running/servers.csv                                3       20
scenarios/small_running/workloads.csv                             79       15
scenarios/small_running/carbon.csv                                96        3
scenarios/small_running/baseline_schedule.csv                     79        5
scenarios/small_running/availability.csv                          12        3


### 5.1 `data/clean/servers.csv`: one row per UniMelb server

All 145 servers are listed (81 R840, 64 C6320), eligible or not, so the reason a server was
left out can be read from its row. Only rows with `eligible = 1` can be drawn into a
scenario, and only for a fleet row of the same model.

| Column | Meaning | Relevance |
|---|---|---|
| `server` | UniMelb server name | identifier |
| `model` | `R840` or `C6320` | which fleet rows the server can fill |
| `n_rows` | 10-minute records kept for the server | evidence behind every other value |
| `completeness` | share of the kept period the server has records for (0-1) | eligibility gate |
| `peak_cpu_load` | highest CPU load observed (0-100) | eligibility gate |
| `power_load_corr` | correlation between power and CPU load | eligibility gate |
| `n_idle_rows` | records with CPU load ≤ 1 | eligibility gate; the sample behind `P_idle_W` |
| `P_idle_W` | mean power at idle, watts | **model**: $P^{idle}_s$, Eq. (17)-(18) |
| `P_max_W` | highest observed power, watts | **model**: $P^{max}_s$, Eq. (19) |
| `P_fit_full_load_W` | power at 100% load from a linear fit, watts | alternative to `P_max_W` for sensitivity runs. Most servers never reach full load, so the observed maximum can understate it |
| `T0_C` | temperature at the first kept record, °C | **model**: initial temperature $T^0_s$, Eq. (30) |
| `peak_temp_C` | highest temperature observed, °C | context for the thermal limits |
| `mean_inlet_C` | mean inlet air temperature, °C | context for the thermal fit |
| `inlet_source` | `measured` (R840: the server's own inlet) or `d1_room_mean` (C6320: the R840 room mean, section 1.6) | which inlet the thermal fit used |
| `N_cores` | cores: 64 for an R840, 80 for a C6320 | **model**: $N^{cores}_s$, gives processing capability in Eq. (15) |
| `C_s_cores` | CPU capacity in cores | **model**: $C_s$. Equal to `N_cores`, since one vCPU is taken to occupy one core |
| `M_s_MB` | memory capacity, MB: 66,036 for an R840, 197,972 for a C6320 | **model**: $M_s$ |
| `th_a_C` | thermal fit: offset $a$, °C | thermal response (section 1.4) |
| `th_R_C_per_W` | thermal fit: temperature rise per watt $R$, °C/W | thermal response |
| `th_alpha_10min` | thermal fit: carry-over $\alpha$ per 10 minutes, as fitted | source of `th_alpha_5min` |
| `th_alpha_5min` | $\alpha$ per 5-minute model period | thermal response |
| `th_steady_rmse_C` | error of the steady-state fit, °C | quality of the thermal fit |
| `eligible` | 1 if the server passes every gate of section 1.4 (and, for a C6320, the configuration gate of section 1.7) | decides whether the server can be used |

### 5.2 `data/clean/workloads.csv`: one row per workload

The workload catalogue: every UFCG VM that survives section 2, ordered by arrival.

| Column | Meaning | Relevance |
|---|---|---|
| `workload_id` | `w0001`, `w0002`, ... in order of arrival | identifier; links to `workload_usage.parquet` and the scenario files |
| `ufcg_server_id` | the VM's id in the raw UFCG files | traces the workload back to the raw data |
| `flavor_name` | OpenStack flavor | source of the two columns that follow |
| `C_req_vcpu` | requested vCPUs | **model**: $C^{req}_i$ |
| `M_req_MB` | requested memory, MB | **model**: $M^{req}_i$ |
| `arrival_unix` | first slot the VM is seen, Unix seconds (UTC) | true arrival time |
| `arrival_slot_of_day` | 5-minute slot of the local day, 0-287 | becomes $A_i$ when arrivals are pooled into one day |
| `lifetime_slots` | slots from first to last sighting | how long the VM existed |
| `n_work_slots` | slots in the work window $T^W_i$ (observed or filled) | lower than `lifetime_slots` when long gaps were left out |
| `n_outage_slots` | slots filled with the VM's mean during collector outages | how much of $W_i$ is estimated |
| `mean_util` | mean usage as a share of the requested vCPUs (0-1) | shows how lightly most VMs are used |
| `peak_util` | highest usage as a share of the requested vCPUs (0-1) | same |
| `core_seconds` | total CPU time used, in seconds of one fully busy core | $W_i$ before conversion to MHz |
| `W_MHz_s` | total work, MHz·s | **model**: $W_i$, Eq. (14) |
| `ET_base_periods` | periods needed on the requested cores at $f_{ref}$ | **model**: $ET^{base}_i$; also used to set deadlines |
| `usage_imputed` | 1 if the usage was borrowed (section 2.6) | lets imputed workloads be excluded or reported separately |
| `usage_donor_ufcg_id` | VM the usage was borrowed from; empty otherwise | traces an imputed value |

### 5.3 `data/clean/workload_usage.parquet`: one row per workload and 5-minute slot

The usage series behind each $W_i$. Slots left out as long gaps (section 2.5) have no
row. It is stored as Parquet because of its size, and opens with
`pandas.read_parquet` or DuckDB.

| Column | Meaning | Relevance |
|---|---|---|
| `workload_id` | links to `workloads.csv` | identifier |
| `k` | slots since the workload arrived (0 = arrival slot) | time index within the workload's life |
| `cpu_pct` | CPU usage in percent of one vCPU, so up to 100 × vCPUs | the cleaned reading |
| `U_cpu_MHz` | the same usage in MHz | **model**: $U^{CPU}_{it}$ |
| `filled` | true if the value was not observed | quick filter for measured values only |
| `fill` | `observed`, `interpolated`, `outage_mean` or `imputed` | how the value was obtained |

### 5.4 `data/clean/cleaning_log_servers.csv` and `cleaning_log_workloads.csv`: one row per cleaning step

| Column | Meaning |
|---|---|
| `dataset` | `UniMelb R840`, `UniMelb C6320` or `UFCG` |
| `step` | the cleaning step, in the order applied |
| `unit` | what is counted: `rows`, `servers` or `VMs` |
| `before`, `after`, `removed` | count entering the step, count leaving it, and the difference |
| `note` | threshold or remark for the step |

These are the counts to quote when reporting how the data was reduced.

### 5.5 `scenarios/<name>/`: the model inputs

Periods are numbered from 1. Columns copied unchanged from the cleaned tables
(`P_idle_W`, `P_max_W`, `P_fit_full_load_W`, `T0_C`, `N_cores`, `C_s_cores`, `M_s_MB`,
the `th_` columns, `usage_imputed`) keep the meaning given above.

**`servers.csv`**: one row per server in the fleet.

| Column | Meaning | Relevance |
|---|---|---|
| `s` | server index | **model**: index $s$ |
| `server` | UniMelb server name | links to the cleaned table |
| `model` | `R840` or `C6320` | the fleet row's server model |
| `cpu` | processor taken for the frequency: Xeon Gold 6130 (2.1 GHz), Xeon Gold 6148 (2.4 GHz), Xeon E5-2690 v4 (2.6 GHz) | source of `f_s_MHz` |
| `zone` | cooling zone | **model**: zone membership (assumed) |
| `f_s_MHz` | CPU frequency | **model**: $f_s$ (assumed: from SPEC CPU results for the model's processor) |
| `PC_s_MHz` | `N_cores` × `f_s_MHz` | **model**: processing capability $PC_s$, Eq. (15) |
| `CT_s_cold_plate` | 1 if the server has a cold plate (the C6320s) | **model**: $CT_s$ (assumed) |
| `n_cold_plates` | cold plates installed on the server, 0.3 kW each; edit to change the count per server | **model**: plates per server when cold plates are modelled (assumed; equals `CT_s_cold_plate` as written) |
| `W_per_MHz` | dynamic watts per MHz delivered; lower is more efficient | shows how much the servers differ in efficiency |
| `U_ceiling_inlet_min`, `U_ceiling_inlet_max` | highest steady utilization that keeps the server within its temperature limit at the coldest and the warmest CRAC supply temperature | shows whether the thermal limit can bind for this server |

**`workloads.csv`**: one row per workload in the scenario.

| Column | Meaning | Relevance |
|---|---|---|
| `i` | workload index, in order of arrival | **model**: index $i$ |
| `workload_id`, `flavor` | as in the cleaned catalogue | links to the cleaned table |
| `A_i` | first period the workload may run | **model**: arrival $A_i$ |
| `D_i` | period by which it must be finished | **model**: deadline $D_i$ (assumed) |
| `AD_i` | `D_i` − `A_i` − `ET_base_i` | **model**: allowable delay $AD_i$, Eq. (22) |
| `ET_base_i` | baseline execution time, periods | **model**: $ET^{base}_i$ |
| `RT_i` | 1 if time-sensitive, 0 if delay-tolerant | **model**: $RT_i$ (assumed) |
| `B_i` | 1 if already started; 0 for every workload | **model**: $B_i$ (assumed) |
| `X0_s` | server in the reference schedule | **model**: initial assignment $X^0_{is}$ |
| `C_req_i`, `M_req_i_MB`, `W_i_MHz_s` | requested vCPUs, requested memory, total work | **model**: $C^{req}_i$, $M^{req}_i$, $W_i$ |

**`carbon.csv`**: one row per period.

| Column | Meaning | Relevance |
|---|---|---|
| `t` | period | **model**: index $t$ |
| `hour_local` | local hour at the middle of the period | readability |
| `CI_t_g_per_kWh` | grid carbon intensity | **model**: $CI_t$ (annual mean from data, daily shape assumed) |

**`baseline_schedule.csv`**: one row per workload, the reference schedule.

| Column | Meaning | Relevance |
|---|---|---|
| `i`, `s` | workload and the server it runs on | the reference assignment |
| `start`, `end` | first period it runs, and the period by which it is finished | the reference timing |
| `late` | 1 if it finishes after `D_i` or could not be placed | whether the reference schedule itself is feasible |

**`params.json`** holds the scenario settings, the model constants, the time
conventions, the list of assumed inputs (`assumed_not_from_data`) and the summary
figures printed in section 3 (`diagnostics`).